# Project — modelling notebook

**Author:** Andrea Gutiérrez Arcas · **Dataset:**  [NBA Dataset: Box Scores and Stats (1947 - Today)](https://www.kaggle.com/datasets/eoinamoore/historical-nba-data-and-player-box-scores) · **Last run:** *PENDIENTE*



## 0. Executive summary

> 📝 **Write this last.** Five lines for the stakeholder, no jargon:
> 1. The question the model answers, and for whom.
> 2. The model you chose, in one phrase.
> 3. How much better it is than today's rule, in the stakeholder's units.
> 4. What the stakeholder should do with it.
> 5. The main limit: when not to trust it.

*TODO*

## Setup

In [126]:
##Al ejecutar los import me ha dado error porque no tengo instalada la librería plotly.

%pip install plotly scikit-learn pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [127]:
from pathlib import Path

import joblib  # noqa: F401
import numpy as np  # noqa: F401
import pandas as pd
import plotly.express as px  # noqa: F401
import sklearn
from sklearn.compose import ColumnTransformer  # noqa: F401
from sklearn.dummy import DummyClassifier, DummyRegressor  # noqa: F401
from sklearn.ensemble import (  # noqa: F401
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer  # noqa: F401
from sklearn.inspection import permutation_importance  # noqa: F401
from sklearn.linear_model import LinearRegression, LogisticRegression  # noqa: F401
from sklearn.metrics import (  # noqa: F401
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (  # noqa: F401
    GridSearchCV,
    RandomizedSearchCV,
    TimeSeriesSplit,
    cross_val_score,
)
from sklearn.pipeline import Pipeline  # noqa: F401
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # noqa: F401

# Use RANDOM_STATE wherever a random_state exists, so two runs give the same numbers.
RANDOM_STATE = 42
# Relative to this notebook (project/), or a URL string.
DATA_PATH = Path("data") / "nba_load.parquet" #ruta a mi notebook de limpieza
MODEL_PATH = Path("model.joblib")

print("pandas", pd.__version__, "· scikit-learn", sklearn.__version__)

pandas 3.0.1 · scikit-learn 1.9.1


## 1. From the brief to an ML task


| Question | Your answer |
|---|---|
| Decision the prediction feeds | *Cuántos minutos juega cada jugador de los Boston Celtics en el próximo partido: con normalidad, con límite de minutos o descanso* |
| Task type (regression / classification) | *Clasificación binaria: lesión nueva sí/no* |
| Target: exact column or formula |*`lesion_nueva` = 1 si el jugador jugó el partido anterior de su equipo y se pierde el siguiente por lesión (`comment` clasificado como lesión); 0 en caso contrario* |
| Grain: what is one prediction? | *Un jugador antes de un partido* |
| When is the prediction made, and how far ahead? | *La víspera de cada partido, solo para ese partido. Tras jugarlo, sus datos se añaden y se predice el siguiente* |
| Features available at that moment | *Carga de los 14 días previos (hasta el último partido jugado), minutos del último partido, días de descanso hasta el próximo partido, si es back-to-back, edad, si es titular habitual y lesiones anteriores en la temporada* |
| Today's rule (the baseline to beat) | *El semáforo del brief: alerta si la carga de 14 días supera el percentil 80 del propio jugador* |
| Model metric, and why | *Recall y PR-AUC. Las lesiones nuevas son raras, así que la *accuracy* engaña. Dejar jugar a alguien que se lesiona cuesta más que un descanso de más, pero los descansos también tienen coste (victorias, normativa)* |
| Business translation (which KPI moves) | *KPI de ausencias por lesión y de disponibilidad de titulares* |

**Changes since the brief:**
- **El horizonte de la predicción.** El brief planteaba la carga de 14 días como semáforo. Ahora el modelo predice el riesgo para el **próximo partido** y se actualiza después de cada uno, que es como el responsable de rendimiento toma la decisión.
- **La variable objetivo es una lesión *nueva*.** Si el jugador ya estaba lesionado, predecir que no jugará no aporta nada al responsable de rendimiento. Esas filas se excluyen del entrenamiento.
- **Entreno con toda la liga y evalúo en los Celtics.** Las lesiones nuevas por partido son poco frecuentes, y con un solo equipo habría pocos casos para aprender. El modelo se entrena con los partidos de todos los equipos y se evalúa por separado en los Celtics, que siguen siendo el club piloto. Los datos se separan por tiempo: se entrena con las temporadas de 2015-16 a 2020-21, se valida con 2022-23 y 2023-24 y se prueba con las dos más recientes, 2024-25 y 2025-26. La temporada 2021-22 no se usa porque la fuente no guarda los motivos de las ausencias, así que no se sabe quién se lesionó (detalle en la limpieza y en el apartado 3).
- **De predecir minutos a predecir riesgo.** El objetivo inicial era recomendar cuántos minutos debe jugar cada jugador. Los datos solo muestran los minutos que dio el entrenador, no los que habrían sido mejores, así que un modelo de minutos aprendería a copiarle. Por eso el modelo predice el **riesgo de lesión**, y una regla de decisión lo convierte en una recomendación de minutos (normal / limitar / descansar).
- **Definición más estricta del target (13/11):** para que una lesión cuente como nueva, el jugador tiene que haber jugado el partido anterior de su equipo. Así evito contar como nuevas lesiones que ya venían de antes y no estaban registradas (detalle en el apartado 3).

**What this means for the stakeholder.** 
Antes de cada partido, la herramienta estima el riesgo de lesión de cada jugador y lo traduce en una recomendación de minutos: jugar con normalidad, limitar minutos o descansar. El objetivo es llegar con la plantilla sana a los playoffs, que es donde se decide el título. El modelo aprende de lo que les ha pasado a los jugadores de toda la liga, pero se aplica y se evalúa en la plantilla de los Celtics, y se actualiza después de cada partido.

## 2. Data and quality

**Ficha del dataset:** [NBA Dataset: Box Scores and Stats (1947 - Today)](https://www.kaggle.com/datasets/eoinamoore/historical-nba-data-and-player-box-scores), publicado en Kaggle por Eoin A. Moore con datos de NBA.com, licencia CC0 (dominio público). Versión 515, descargada el 14/10/2026. **Grano:** una fila por jugador y partido. **Periodo:** desde 1947 hasta la temporada 2025-26; yo uso desde la temporada 2015-16.

La limpieza está en `notebooks/02-limpieza.ipynb`, que parte de los CSV originales (no están en GitHub porque pesan 390 MB) y guarda `data/nba_load.parquet` (3,9 MB). Aquí cargo ese archivo y compruebo que está como espero.

In [128]:
#PRIMERO: cargo el dataset limpio que guardé en 02-limpieza
df = pd.read_parquet(DATA_PATH)
print(f"Filas: {len(df):,} · Columnas: {df.shape[1]}")
print("Periodo:", df["gameDate"].min().date(), "→", df["gameDate"].max().date())

#SEGUNDO: compruebo que no hay jugadores repetidos en el mismo partido
#Cada jugador solo puede aparecer una vez por partido; si aparece dos veces, su carga se contaría doble
duplicated_player_game = df.duplicated(subset=["personId", "gameId"]).sum()
print("Filas duplicadas (mismo jugador y partido):", duplicated_player_game)

#TERCERO: nulos que quedan en cada columna
null_counts = df.isna().sum()
print("\nNulos por columna (solo las que tienen alguno):")
print(null_counts[null_counts > 0])

#CUARTO: tipos de datos, para comprobar que el parquet ha conservado fechas y números
df.dtypes

Filas: 366,218 · Columnas: 19
Periodo: 2015-10-27 → 2026-06-13
Filas duplicadas (mismo jugador y partido): 0

Nulos por columna (solo las que tienen alguno):
startingPosition    214516
comment             307623
edad                    11
dtype: int64


personId                        int64
firstName                         str
lastName                          str
gameId                          int64
gameDate               datetime64[us]
gameType                          str
season_start                    int32
comments_available               bool
playerteamName                    str
opponentteamName                  str
home                          float64
numMinutes                    float64
startingPosition                  str
points                        float64
fieldGoalsAttempted           float64
freeThrowsAttempted           float64
comment                           str
motivo_ausencia                   str
edad                          float64
dtype: object

**De 1.669.922 filas a 366.218.** Cada paso, con sus filas y el motivo:

| Step | Rows affected | Decision and reason |
|---|---|---|
| Partidos anteriores a la temporada 2015-16 | 1.273.392 eliminadas | Fuera de mi ventana de trabajo |
| Pretemporada, All-Star y otros | 30.312 eliminadas | No son competición real. Mantengo temporada regular, playoffs, Play-in y NBA Cup porque son oficiales y generan carga |
| Filas sin jugador (`personId` vacío) | 0 | Control: no se pueden asignar a ningún jugador |
| Minutos en formato mm:ss | 0 | Control: se convertirían a decimal |
| Duplicados (mismo jugador y partido) | 0 | Comprobado al cargar: cada jugador aparece una vez por partido |
| Minutos vacíos (no jugó) | 64.565 rellenadas con 0 | Ese día su carga fue 0; la media inventaría minutos y borrarlas quitaría las ausencias |
| Valores atípicos en minutos | 0 | El máximo (64 min) es posible: un partido con tres prórrogas. No hay negativos |
| Comentarios clasificados en `motivo_ausencia` | 58.595 | Por palabras clave: lesion, decision_tecnica u otros. lesion es la base del target |
| Protocolo COVID fuera de lesion | 67 | Un contagio no depende de la carga |
| Sin minutos y sin comentario (`sin_motivo`) | 8.392 | No jugó y no se sabe por qué: ni lesión ni partido jugado |
| Temporada 2021-22 sin comentarios | 33.848 marcadas | Se mantienen para la carga y H2, pero no se usan como target |
| Sin fecha de nacimiento | 11 | Se mantienen; el pipeline rellenará la edad con la mediana |

**What this means for the stakeholder.** Trabajo con diez temporadas completas de la NBA (2015-16 a 2025-26): más de 366.000 registros de jugador y partido, con los minutos jugados y, cuando un jugador no juega, el motivo. Los datos son fiables para medir la carga de minutos de cada jugador. Para saber quién se lesionó hay una limitación: en la temporada 2021-22 la fuente no guarda el motivo de las ausencias, así que esa temporada no la uso para aprender qué provoca las lesiones.

## 3. What the exploration showed

### Construcción del target

> **Registro de cambios del target (13/11/2026)**
>
> **Qué ha cambiado:** he añadido la decisión 5. Ahora una fila solo está en riesgo si el último partido del jugador es el **partido anterior de su equipo**. Antes bastaba con que el jugador hubiera jugado su último partido registrado.
>
> **Por qué:** al construir la feature de días de descanso (apartado 4), los jugadores con 8 días o más sin jugar tenían cuatro veces más "lesiones nuevas" que el resto. Al investigarlo vi que eran jugadores que desaparecen de los datos durante varios partidos de su equipo y vuelven a aparecer como lesionados: lesiones que probablemente venían de antes y no estaban registradas.
>
> **Efecto en los resultados:**
>
> | | Antes | Después |
> |---|---|---|
> | Filas en riesgo | 267.599 | 257.888 (−9.711) |
> | Lesiones nuevas | 3.518 | 2.905 (−613) |
> | Tasa de lesión nueva | 1,31 % | 1,13 % |
>
> Las 613 lesiones quitadas eran el 17 % del total. Todos los números de los apartados 3 y 4 están calculados con el target nuevo.
>
> **Qué ha mejorado:** la relación entre carga y lesión es ahora más clara (el grupo con más carga de 14 días tiene casi el triple de riesgo que el de menos), y los días de descanso muestran lo esperable: a más descanso, menos lesiones.

Antes de explorar necesito el target. Lo construyo aquí, y no con las features de la §4, porque forma parte de la definición del problema y la exploración lo necesita.

Para construirlo he tenido que tomar cuatro decisiones:

**Decisión 1: solo cuento las lesiones nuevas**

Cuando un jugador se lesiona, aparece como "lesion" en todos los partidos que se pierde. Si se pierde 10 partidos por un esguince, tiene 10 filas "lesion", pero solo la primera es el momento en que se lesiona. Las otras 9 son la misma lesión que continúa.

- **Alternativa descartada: usar todas las filas "lesion" como target.** El modelo aprendería que "si ayer estaba lesionado, hoy también", que acertaría mucho pero no serviría para nada: el cuerpo técnico ya sabe que un jugador lesionado no puede jugar. Lo que necesita saber es quién está a punto de lesionarse.
- **Decisión:** `lesion_nueva` = 1 si el jugador jugó el partido anterior y se pierde este por lesión.

**Decisión 2: solo entran en el modelo los jugadores que jugaron el partido anterior**

La pregunta "¿se lesionará en el próximo partido?" solo tiene sentido para un jugador que está sano y jugando. Si en el partido anterior ya estaba lesionado, descansó o no jugó por otro motivo, no está en riesgo de lesionarse jugando.

- **Alternativa descartada: incluir todas las filas y poner 0 en las que no son lesión nueva.** Mezclaría jugadores sanos con jugadores que ya estaban lesionados o no jugaron, y el modelo aprendería de casos en los que la pregunta no tiene sentido.
- **Decisión:** las filas en las que el jugador no jugó el partido anterior se quedan en el dataset (sus minutos cuentan para calcular la carga), pero el target queda vacío y no entran en el modelo.
- **Limitación:** una lesión que llega justo después de un descanso no cuenta como nueva. Lo acepto porque mi modelo mide la carga de los minutos jugados, y en ese caso el último partido no lo jugó.

**Decisión 3: el partido anterior tiene que ser de la misma temporada**

En el primer partido de cada temporada, el "partido anterior" fue meses antes, en la temporada pasada.

- **Alternativa descartada: contar también el primer partido de la temporada.** Una lesión en ese partido puede venir del verano o de la pretemporada, no de la carga de minutos, y la carga de los últimos 14 días sería 0 porque no ha habido partidos. Serían casos que mi modelo no puede explicar.
- **Decisión:** solo cuento como en riesgo las filas cuyo partido anterior es de la misma temporada.

**Decisión 4: la temporada 2021-22 no tiene target**

En la limpieza vi que esa temporada no tiene comentarios, así que no se sabe quién se lesionó (ver `02-limpieza.ipynb`).

- **Alternativa descartada: poner 0 en todas sus filas.** Sería decir que en toda una temporada nadie se lesionó, y el modelo aprendería algo falso.
- **Decisión:** sus filas se quedan para calcular la carga, pero no tienen target (`comments_available = False`).

**Decisión 5: el jugador tiene que haber jugado el partido anterior de su equipo**

Hay jugadores que desaparecen de los datos durante varios partidos de su equipo y vuelven a aparecer como lesionados. Si su última fila era "jugó", la lesión parecería nueva, cuando lo más probable es que llevara días lesionado sin registrar. Lo descubrí al construir los días de descanso, y la prueba con datos está justo debajo del código.

- **Alternativa descartada: quitar solo los huecos largos (8 días o más).** El umbral sería arbitrario: la tasa de "lesión" es alta incluso con huecos de 2 o 3 días.
- **Decisión:** una fila solo está en riesgo si el último partido del jugador es el partido anterior de su equipo.

**En resumen:** una fila está **en riesgo** (y entra en el modelo) si el jugador jugó el partido anterior de su equipo, ese partido es de la misma temporada y la temporada tiene comentarios. Las demás filas se quedan en el dataset para calcular la carga, pero con el target vacío.

In [129]:
#PRIMERO: busco el partido anterior de cada EQUIPO
#Así puedo comprobar si el jugador jugó el partido anterior de su equipo o si faltan partidos en sus datos
team_games = (
    df[["playerteamName", "gameId", "gameDate"]]
    .drop_duplicates(["playerteamName", "gameId"])
    .sort_values(["playerteamName", "gameDate"])
)
team_games["team_prev_game"] = team_games.groupby("playerteamName")["gameId"].shift(1)

#Lo uno a cada fila (borro antes la columna si ya existe, para que la celda se pueda repetir)
n = len(df)
df = df.drop(columns=["team_prev_game"], errors="ignore")
df = df.merge(team_games[["playerteamName", "gameId", "team_prev_game"]], on=["playerteamName", "gameId"], how="left")
print("Filas antes:", n, "| después:", len(df))   # deben ser IGUALES

#SEGUNDO: ordeno por jugador y fecha para que "el partido anterior" sea el anterior de ese jugador
#Va después del merge porque el merge puede cambiar el orden de las filas
df = df.sort_values(["personId", "gameDate"]).reset_index(drop=True)

#TERCERO: para cada fila, miro qué pasó en el partido anterior del mismo jugador
#shift(1) dentro de cada jugador trae el valor de su fila anterior; en su primer partido queda vacío
by_player = df.groupby("personId")
df["prev_status"] = by_player["motivo_ausencia"].shift(1)
df["prev_season"] = by_player["season_start"].shift(1)
df["prev_game"] = by_player["gameId"].shift(1)

#CUARTO: condiciones para estar "en riesgo"
played_prev_game = df["prev_status"] == "jugó"
same_season = df["prev_season"] == df["season_start"]
played_team_prev_game = df["prev_game"] == df["team_prev_game"]   #su último partido es el anterior de su equipo
df["at_risk"] = played_prev_game & same_season & played_team_prev_game & df["comments_available"]

#QUINTO: el target, solo en las filas en riesgo (en las demás queda vacío)
df["lesion_nueva"] = (df["motivo_ausencia"] == "lesion").astype("Int64").where(df["at_risk"])

#SEXTO: compruebo por qué queda fuera cada fila (los grupos deben sumar el total)
exclusion_summary = pd.Series({
    "Primer partido del jugador en el dataset": df["prev_status"].isna().sum(),
    "No jugó el partido anterior": (df["prev_status"].notna() & ~played_prev_game).sum(),
    "Partido anterior en otra temporada": (played_prev_game & ~same_season).sum(),
    "Faltan partidos de su equipo en sus datos": (played_prev_game & same_season & ~played_team_prev_game).sum(),
    "Temporada sin comentarios (2021-22)": (played_prev_game & same_season & played_team_prev_game & ~df["comments_available"]).sum(),
    "En riesgo: filas del modelo": df["at_risk"].sum(),
})
print(exclusion_summary)
print("Suma:", exclusion_summary.sum(), "| Total filas:", len(df))

#SÉPTIMO: compruebo la decisión 1: qué pasó en el partido anterior de cada fila "lesion"
print("\nPartido anterior de las filas 'lesion':")
print(df.loc[df["motivo_ausencia"] == "lesion", "prev_status"].value_counts(dropna=False))

#OCTAVO: porcentaje de lesiones nuevas
new_injuries = int(df["lesion_nueva"].sum())
rows_at_risk = int(df["at_risk"].sum())
injury_rate = new_injuries / rows_at_risk
print(f"\nLesiones nuevas: {new_injuries:,} de {rows_at_risk:,} filas en riesgo = {injury_rate:.2%}")

Filas antes: 366218 | después: 366218
Primer partido del jugador en el dataset       1582
No jugó el partido anterior                   66335
Partido anterior en otra temporada             3339
Faltan partidos de su equipo en sus datos     11035
Temporada sin comentarios (2021-22)           26039
En riesgo: filas del modelo                  257888
dtype: int64
Suma: 366218 | Total filas: 366218

Partido anterior de las filas 'lesion':
prev_status
lesion              4050
jugó                3564
decision_tecnica     380
otros                 62
sin_motivo            33
NaN                   24
Name: count, dtype: int64

Lesiones nuevas: 2,905 de 257,888 filas en riesgo = 1.13%


**Resultado:** de 366.218 filas, 257.888 están en riesgo y forman la tabla del modelo. Hay 2.905 lesiones nuevas, el 1,13 % de las filas en riesgo.

- La decisión 1 era necesaria: de las 8.113 filas "lesion", 4.050 vienen después de otra fila "lesion". Son la misma lesión que continúa. Si las contara, el modelo aprendería que "si ayer estaba lesionado, hoy también", que no sirve para decidir nada.
- **Limitación:** hay 380 lesiones que llegan después de un descanso del entrenador. No cuentan como nuevas porque el jugador no jugó el partido anterior, y mi modelo mide la carga de minutos jugados.
- **Con un 1,13 % de positivos, la clase está muy desbalanceada.** Un modelo que dijera siempre "no se lesiona" acertaría el 98,87 % de las veces y no serviría para nada. Por eso uso recall y PR-AUC, y no accuracy, como decidí en el apartado 1.

In [130]:
#Comprobación de la decisión 5: ¿qué pasa con las filas en las que faltan partidos de su equipo?
#Parto de las filas que cumplen todo lo demás (jugó su último partido, misma temporada, temporada con comentarios)
#Son exactamente las filas en riesgo que tenía ANTES de añadir la decisión 5
candidates = df[played_prev_game & same_season & df["comments_available"]].copy()
candidates["missed_team_games"] = ~played_team_prev_game[candidates.index]
candidates["is_injury"] = candidates["motivo_ausencia"] == "lesion"

#PRIMERO: antes y después de la decisión 5
before_rows = len(candidates)
before_injuries = int(candidates["is_injury"].sum())
after_rows = int((~candidates["missed_team_games"]).sum())
after_injuries = int(candidates.loc[~candidates["missed_team_games"], "is_injury"].sum())
removed_injuries = before_injuries - after_injuries
print(f"Antes:   {before_rows:,} filas en riesgo · {before_injuries:,} lesiones nuevas · {before_injuries / before_rows:.2%}")
print(f"Después: {after_rows:,} filas en riesgo · {after_injuries:,} lesiones nuevas · {after_injuries / after_rows:.2%}")
print(f"Lesiones quitadas: {removed_injuries:,} ({removed_injuries / before_injuries:.0%} de las de antes)")

#SEGUNDO: tasa de "lesión" según si faltan partidos del equipo
gap_check = candidates.groupby("missed_team_games")["is_injury"].agg(["size", "sum", "mean"])
gap_check["mean"] = (gap_check["mean"] * 100).round(2)
print("\n", gap_check)

#TERCERO: tasa de "lesión" (%) por días desde el último partido
#Columna "Todas": lo que veía ANTES de la decisión 5; las otras dos separan los tipos de fila
prev_day = df.groupby("personId")["gameDate"].shift(1).dt.normalize()
candidates["days_gap"] = (candidates["gameDate"].dt.normalize() - prev_day[candidates.index]).dt.days
candidates["gap_band"] = pd.cut(candidates["days_gap"], bins=[1, 2, 3, 4, 8, 1000], right=False, labels=["1", "2", "3", "4-7", "8+"])
gap_table = (candidates.groupby(["gap_band", "missed_team_games"], observed=True)["is_injury"].mean() * 100).unstack()
gap_table.columns = ["Jugó el partido anterior del equipo", "Faltan partidos del equipo"]
gap_table.insert(0, "Todas", candidates.groupby("gap_band", observed=True)["is_injury"].mean() * 100)
print("\n", gap_table.round(2))

#CUARTO: ejemplos de "lesiones nuevas" que en realidad venían de antes
#Comparo la fecha de su último partido con la del partido anterior de su equipo
#Cada partido aparece dos veces en team_games (una por equipo); me quedo con una, porque la fecha es la misma
team_prev_day = team_games.drop_duplicates("gameId").set_index("gameId")["gameDate"]
examples = candidates[candidates["missed_team_games"] & candidates["is_injury"]].head(3).copy()
examples["player_last_game"] = prev_day[examples.index].dt.date
examples["team_prev_game_day"] = examples["team_prev_game"].map(team_prev_day).dt.date
examples[["firstName", "lastName", "gameDate", "player_last_game", "team_prev_game_day", "comment"]]

Antes:   267,599 filas en riesgo · 3,518 lesiones nuevas · 1.31%
Después: 257,888 filas en riesgo · 2,905 lesiones nuevas · 1.13%
Lesiones quitadas: 613 (17% de las de antes)

                      size   sum  mean
missed_team_games                    
False              257888  2905  1.13
True                 9711   613  6.31

           Todas  Jugó el partido anterior del equipo  \
gap_band                                               
1          1.80                                 1.80   
2          1.04                                 1.04   
3          1.02                                 0.86   
4-7        2.39                                 0.73   
8+         4.12                                 0.61   

          Faltan partidos del equipo  
gap_band                              
1                               0.00  
2                               6.25  
3                               4.67  
4-7                             6.31  
8+                              7.06  


,firstName,lastName,gameDate,player_last_game,team_prev_game_day,comment
146,Tim,Duncan,2016-02-09 20:00:00,2016-01-22,2016-02-06,DND - Sore Right Knee
1361,Jason,Terry,2017-12-15 20:00:00,2017-12-04,2017-12-13,DND - Injury/Illness
2461,Tyson,Chandler,2018-02-10 21:00:00,2018-02-06,2018-02-07,DND - Injury/Illness


**Prueba de la decisión 5.** La descubrí al preparar la feature de días de descanso (apartado 4). En la columna "Todas" de la tabla por días se ve el problema tal como apareció: los jugadores que llevaban 8 días o más sin jugar tenían un 4,12 % de "lesiones nuevas", cuatro veces más que los que llevaban 2 o 3 días (alrededor del 1 %). No tiene sentido, porque estaban descansados.

Al separar las filas según si el jugador jugó o no el partido anterior de su equipo:
- **Cuando faltan partidos del equipo**, la tasa de "lesión" es del 6,31 % (613 de 9.711 filas), frente al 1,13 % del resto. Son solo el 3,6 % de las filas, pero se llevaban el 17 % de las lesiones nuevas (613 de 3.518).
- **En las filas sin huecos**, la tasa baja a medida que aumenta el descanso (del 1,80 % con un día al 0,61 % con ocho o más), que es lo esperable. En las filas con huecos es alta sea cual sea el descanso (entre el 4,67 % y el 7,06 %). La fila de un día vale 0 porque es casi imposible: si el jugador jugó ayer, no ha podido perderse partidos de su equipo entre medias.

Los ejemplos lo muestran con fechas. Tim Duncan jugó su último partido el 22/01/2016, su equipo siguió jugando sin él hasta el 06/02/2016 (más de dos semanas), y el 09/02 aparece como "Sore Right Knee" (molestias en la rodilla derecha). Esa lesión no empezó ese día. Lo mismo pasa con Jason Terry y Tyson Chandler.

In [131]:
#Ejemplo: la racha de partidos seguidos con lesión más larga de un jugador
#Cada vez que el motivo cambia, empieza una "racha" nueva; cuento cuántos partidos dura cada una
injury_row = df["motivo_ausencia"] == "lesion"
streak_id = (df["motivo_ausencia"] != df["prev_status"]).cumsum()
streak_length = injury_row.groupby(streak_id).transform("sum")

#Me quedo con la racha de lesión más larga y enseño sus partidos, más el anterior (cuando aún jugaba)
longest = streak_id[injury_row & (streak_length == streak_length[injury_row].max())].iloc[0]
rows = df.index[streak_id == longest]
example = df.loc[rows.min() - 1 : rows.max(), ["firstName", "lastName", "gameDate", "motivo_ausencia", "prev_status", "lesion_nueva"]]
print(f"Racha de lesión más larga: {len(rows)} partidos")
example.head(6)

Racha de lesión más larga: 42 partidos


,firstName,lastName,gameDate,motivo_ausencia,prev_status,lesion_nueva
213918,Zach,Collins,2020-08-15 14:30:00,jugó,jugó,0
213919,Zach,Collins,2020-08-18 21:00:00,lesion,jugó,1
213920,Zach,Collins,2020-08-20 21:00:00,lesion,lesion,<NA>
213921,Zach,Collins,2020-08-22 20:30:00,lesion,lesion,<NA>
213922,Zach,Collins,2020-08-24 21:00:00,lesion,lesion,<NA>
213923,Zach,Collins,2020-08-29 21:00:00,lesion,lesion,<NA>


**Ejemplo:** en la racha de lesión más larga del dataset, el jugador se pierde 42 partidos seguidos. Solo el primero cuenta como lesión nueva (el partido anterior lo jugó); en los demás, el partido anterior también fue "lesion", así que es la misma lesión que continúa y no entra en el modelo.

### EDA del Target

In [132]:
#PRIMERO: creo los grupos de edad (los usaré aquí y en la comparación de errores por grupo)
#Menores de 25 (jóvenes, contratos de rookie), de 25 a 29 (plenitud) y 30 o más (veteranos)
df["grupo_edad"] = pd.cut(df["edad"], bins=[0, 25, 30, 100], right=False, labels=["<25", "25-29", "30+"])

#SEGUNDO: la tabla del modelo son solo las filas en riesgo (las que tienen target)
model_rows = df[df["at_risk"]].copy()

#TERCERO: gráfico de la distribución del target
target_counts = model_rows["lesion_nueva"].map({0: "Sin lesión nueva", 1: "Lesión nueva"}).value_counts().reset_index()
target_counts.columns = ["outcome", "rows"]
target_counts["pct"] = (target_counts["rows"] / target_counts["rows"].sum() * 100).round(2)

fig = px.bar(
    target_counts, x="outcome", y="rows", text="pct",
    title="Distribución del target en las filas en riesgo",
    labels={"outcome": "", "rows": "Filas (jugador y partido)"},
)
fig.update_traces(texttemplate="%{text} %")
fig.show()

**Lectura:** de las 254.983 filas en riesgo, solo 2.905 son lesiones nuevas (1,13 %). Es un problema muy desbalanceado: por cada lesión nueva hay unas 88 filas sin lesión.

**Qué supone para el modelo:**
- Un modelo que dijera siempre "no se lesiona" acertaría el 98,87 % de las veces sin detectar ninguna lesión. Por eso no uso accuracy, sino **recall** (cuántas lesiones detecto) y **PR-AUC** (lo bien que separa lesiones de no lesiones sin depender del umbral).
- En PR-AUC, un modelo que no sabe nada saca 0,011, que es la proporción de lesiones. Ese es el valor que hay que superar, no 0,5.
- Uso `class_weight="balanced"` para que el modelo dé más peso a los pocos casos de lesión y no los ignore.

In [133]:
#Tasa de lesión nueva por temporada (en %): lesiones nuevas / filas en riesgo de esa temporada
season_rate = model_rows.groupby("season_start")["lesion_nueva"].agg(["mean", "size"]).reset_index()
season_rate.columns = ["season_start", "injury_rate", "rows"]
season_rate["injury_rate"] = season_rate["injury_rate"] * 100
#Nombre de la temporada tipo "2015-16"
season_rate["season"] = season_rate["season_start"].astype(str) + "-" + (season_rate["season_start"] + 1).astype(str).str[-2:]

fig = px.bar(
    season_rate, x="season", y="injury_rate",
    title="Tasa de lesión nueva por temporada (2021-22 sin datos de motivos)",
    labels={"season": "Temporada", "injury_rate": "Lesiones nuevas (% de filas en riesgo)"},
)
fig.show()

**Lectura:** la tasa de lesión nueva no es estable. Hasta 2019-20 está alrededor del 0,9 % (entre 0,80 % y 1,00 %). Hay dos subidas:
- **2020-21 (1,44 %):** fue la temporada del COVID, con 72 partidos concentrados en menos meses de lo habitual (de diciembre a mayo). Más partidos en menos días es justo lo que plantea H1, así que es una pista a favor de que la carga influye.
- **2024-25 y 2025-26 (1,65 % y 1,63 %):** las dos temporadas más altas. Con estos datos no puedo saber si es porque hay más lesiones o porque se registran de otra forma.

**Qué supone para el modelo:** el test (2024-26) tiene bastantes más lesiones que el entrenamiento (alrededor del 0,9-1 %). El modelo aprende con un mundo en el que hay menos lesiones de las que encontrará en el test. Es una situación realista (el futuro no siempre se parece al pasado), pero hay que tenerla en cuenta:
- Al comparar resultados de distintos periodos, la PR-AUC hay que leerla frente a la tasa de lesiones de cada periodo.
- El umbral que elija en validación (2022-24) podría dar más alertas en el test, porque allí hay más lesiones.

In [134]:
#Tasa de lesión nueva por mes, en el orden de la temporada (de octubre a junio)
month_order = [10, 11, 12, 1, 2, 3, 4, 5, 6]
month_names = {10: "Oct", 11: "Nov", 12: "Dic", 1: "Ene", 2: "Feb", 3: "Mar", 4: "Abr", 5: "May", 6: "Jun"}

month_rate = model_rows.groupby(model_rows["gameDate"].dt.month)["lesion_nueva"].agg(["mean", "size"]).reindex(month_order).reset_index()
month_rate.columns = ["month", "injury_rate", "rows"]
month_rate["injury_rate"] = month_rate["injury_rate"] * 100
month_rate["month_name"] = month_rate["month"].map(month_names)

fig = px.line(
    month_rate, x="month_name", y="injury_rate", markers=True,
    title="Tasa de lesión nueva a lo largo de la temporada",
    labels={"month_name": "Mes", "injury_rate": "Lesiones nuevas (% de filas en riesgo)"},
)
fig.show()

#Comparo temporada regular y playoffs, porque mayo y junio son casi solo playoffs
print(model_rows.groupby("gameType")["lesion_nueva"].agg(["mean", "size"]).assign(mean=lambda t: (t["mean"] * 100).round(2)))

                mean    size
gameType                    
NBA Cup         0.22    1384
Play-in         0.16     623
Playoffs        0.28   17220
Regular Season   1.2  238661


**Lectura:** el riesgo de lesión nueva crece mes a mes durante la temporada regular: pasa del 0,51 % en octubre al 1,45 % en marzo y al 1,46 % en abril, casi el triple. A medida que se acumulan partidos, los jugadores se lesionan más. Es una primera pista a favor de H1 (la carga acumulada aumenta el riesgo de lesión).

Pero no basta para demostrar H1: en marzo no solo hay más carga acumulada, también hay otras cosas distintas a octubre (equipos que ya no se juegan nada, viajes, etc.). La prueba directa de H1 es comparar el riesgo según la carga de los últimos 14 días de cada jugador, que hago en el apartado 4.

**La caída de mayo y junio** se debe a los playoffs, que son casi los únicos partidos de esos meses. En playoffs la tasa es mucho más baja (0,28 %) que en temporada regular (1,20 %). Puede tener varias causas que no puedo separar con estos datos: en playoffs los jugadores juegan con molestias que en temporada regular les harían descansar, solo juegan los mejores equipos, y muchos lesionados ya venían de antes (no son lesiones nuevas).

**Qué supone para el modelo:**
- El tipo de partido influye en el riesgo, así que en el apartado 4 añadiré una feature que diga si el partido es de playoffs. Se sabe antes del partido, así que no hay fuga de información.
- El modelo aprende sobre todo de la temporada regular (238.661 de las 257.888 filas). Sus predicciones en playoffs serán menos fiables, y es una limitación que el cuerpo técnico debe conocer, porque los playoffs son justo donde más importa tener a los jugadores sanos.

In [135]:
#Tasa de lesión nueva por grupo de edad
age_rate = model_rows.groupby("grupo_edad", observed=True)["lesion_nueva"].agg(["mean", "size"]).reset_index()
age_rate.columns = ["grupo_edad", "injury_rate", "rows"]
age_rate["injury_rate"] = age_rate["injury_rate"] * 100

fig = px.bar(
    age_rate, x="grupo_edad", y="injury_rate", text="rows",
    title="Tasa de lesión nueva por grupo de edad",
    labels={"grupo_edad": "Edad (años)", "injury_rate": "Lesiones nuevas (% de filas en riesgo)"},
)
fig.update_traces(texttemplate="%{text:,} filas")
fig.show()

**Lectura:** el riesgo sube con la edad: 0,97 % en los menores de 25, 1,16 % entre 25 y 29 y 1,35 % en los de 30 o más. Un veterano tiene alrededor de un 40 % más de riesgo que un jugador joven.

**Qué supone para el modelo:**
- La edad es una feature útil y la mantengo.
- Los grupos de edad serán los grupos para comparar errores en el apartado 10 (`GROUP_COL = "grupo_edad"` en el notebook de la lesson 09). Si el modelo funciona peor con algún grupo, el cuerpo técnico debería saberlo antes de usarlo con ese grupo.
- He elegido tres grupos con sentido deportivo (jóvenes con contrato de rookie, plenitud y veteranos) y con suficientes filas cada uno: el más pequeño, 30+, tiene 57.904 filas y 781 lesiones nuevas.

| Hypothesis (from the brief) | Status: supported / refuted / not testable yet | Evidence (chart, table or number) |
|---|---|---|
| H1: más carga de minutos en los últimos 14 días aumenta el riesgo de lesión | **Apoyada** (relación, no causa) | El grupo con más carga de 14 días tiene un 1,54 % de lesiones nuevas, frente al 0,56 % del grupo con menos: casi el triple (apartado 4, feature 1). El poco descanso también aumenta el riesgo: 1,80 % en back-to-back frente a 0,61 % con 8 días o más (features 3 y 4) |
| H2: tras un back-to-back baja la eficiencia (TS%) | **No confirmada** | El TS% solo baja de 56,81 % a 56,31 %, y jugador a jugador el cambio es de −0,22 puntos con un p-valor de 0,16: la diferencia puede ser casualidad (apartado 4, prueba de H2) |

**What this means for the stakeholder.** Las lesiones nuevas son poco frecuentes (algo más de 1 de cada 100 partidos jugados), así que el modelo tiene que encontrar pocas señales entre mucho ruido. Hay tres patrones claros: el riesgo casi se triplica de octubre a marzo a medida que se acumulan partidos, los veteranos de 30 o más años se lesionan más que los jóvenes, y en las dos últimas temporadas hay más lesiones que antes. Estas pistas apoyan la idea de que la carga acumulada importa, y lo compruebo directamente en el siguiente paso.

## 4. Features

Construyo las features una a una. Cada una solo puede usar información que se conoce **la víspera del partido**: si usara algo que solo se sabe después (por ejemplo, los minutos del propio partido), el modelo haría trampa y parecería mejor de lo que es. Para cada feature explico qué mide, cómo la construyo, compruebo con un ejemplo que no usa información del futuro y miro con datos si ayuda a predecir.

### Feature 1: carga de 14 días (`load_14d`)

**Qué mide:** los minutos que el jugador ha jugado en los 14 días anteriores al partido. Es la variable de mi hipótesis H1 y la del semáforo del brief.

**Decisiones:**
- **La ventana termina el día antes del partido.** La víspera solo conozco los partidos ya jugados; si incluyera el propio partido, sumaría minutos que aún no se han jugado.
- **14 días naturales, no un número de partidos.** La fatiga depende de cuántos minutos se acumulan en un tiempo: 5 partidos en 7 días no cansan lo mismo que 5 partidos en 12 días. Uso solo el día del partido, sin la hora, para que la hora de inicio no cambie qué partidos entran en la ventana.
- **Sin partidos en la ventana, la carga es 0.** Si no ha jugado en 14 días, su carga real es cero, no un dato que falta.

**Resultado:** en las filas en riesgo, la carga media es de 124 minutos (mínimo 0, máximo 339). Es razonable: en 14 días se juegan unos 6 o 7 partidos, y 6 partidos de unos 20 minutos suman unos 120.

In [136]:
#PRIMERO: me quedo con el día del partido, sin la hora
#Así la ventana de 14 días no depende de si el partido empezó a las 19:00 o a las 20:30
df["game_day"] = df["gameDate"].dt.normalize()

#SEGUNDO: para cada jugador, sumo los minutos de los 14 días anteriores al partido
#rolling("14D") hace una ventana de 14 días de calendario
#closed="left" deja FUERA el propio partido: solo suma partidos ya jugados (evita la fuga de información)
load_14d = (
    df.set_index("game_day")
    .groupby("personId")["numMinutes"]
    .rolling("14D", closed="left")
    .sum()
)

#TERCERO: lo añado al dataframe
#df está ordenado por jugador y fecha, igual que el resultado del groupby, así que las filas coinciden
df["load_14d"] = load_14d.to_numpy()


#CUARTO: si no hay partidos en la ventana, la suma sale vacía; su carga real es 0
df["load_14d"] = df["load_14d"].fillna(0)

df.loc[df["at_risk"], "load_14d"].describe()

count    257888.000000
mean        123.891511
std          68.525055
min           0.000000
25%          69.000000
50%         124.000000
75%         177.000000
max         338.630000
Name: load_14d, dtype: float64

In [137]:
#COMPRUEBO QUE NO USA INFORMACIÓN DEL FUTURO
#Compruebo con un jugador concreto que la carga solo suma partidos ANTERIORES
#Jayson Tatum (Celtics), primeros partidos de la temporada 2023-24
check_player = df[(df["lastName"] == "Tatum") & (df["season_start"] == 2023)]
check_player[["game_day", "numMinutes", "load_14d"]].head(8)

,game_day,numMinutes,load_14d
209826,2023-10-25,38.0,0.0
209827,2023-10-27,41.0,38.0
209828,2023-10-30,27.0,79.0
209829,2023-11-01,26.0,106.0
209830,2023-11-04,40.0,132.0
209831,2023-11-06,42.0,172.0
209832,2023-11-08,37.0,214.0
209833,2023-11-10,33.0,213.0


**Comprobación de que no hay fuga de información:** en los primeros partidos de Jayson Tatum en 2023-24, la carga de cada partido es la suma de los minutos de los partidos anteriores, nunca la del propio partido:
- El 25/10, su primer partido, la carga es 0, porque no hay partidos anteriores en la ventana.
- El 27/10 la carga es 38, los minutos del 25/10. No incluye los 41 que juega ese mismo día.
- El 10/11 la carga baja de 214 a 213: el partido del 25/10 sale de la ventana porque queda a más de 14 días, y entra el del 08/11 (214 − 38 + 37 = 213).

In [138]:
#¿AYUDA A PREDECIR?¿Y POR QUÉ 14 DÍAS?
#PRIMERO: calculo también la carga con ventanas de 7 y 28 días, solo para comparar (no se guardan como features)
rows_at_risk = df["at_risk"]
window_comparison = []
for days in [7, 14, 28]:
    load = (
        df.set_index("game_day").groupby("personId")["numMinutes"]
        .rolling(f"{days}D", closed="left").sum()
        .to_numpy()
    )
    load = pd.Series(load, index=df.index).fillna(0)[rows_at_risk]

    #SEGUNDO: 5 grupos del mismo tamaño según la carga, y la tasa de lesión de cada uno
    quintile = pd.qcut(load, 5, labels=["Q1 (menos carga)", "Q2", "Q3", "Q4", "Q5 (más carga)"])
    rate = df.loc[rows_at_risk, "lesion_nueva"].groupby(quintile, observed=True).mean() * 100
    window_comparison.append(pd.DataFrame({"quintile": rate.index, "injury_rate": rate.values, "window": f"{days} días"}))

window_comparison = pd.concat(window_comparison)

#TERCERO: gráfico de la tasa de lesión por quintil de carga, para cada ventana
fig = px.line(
    window_comparison, x="quintile", y="injury_rate", color="window", markers=True,
    title="Tasa de lesión nueva según la carga previa (filas en riesgo)",
    labels={"quintile": "Grupo de carga", "injury_rate": "Lesiones nuevas (%)", "window": "Ventana"},
)
fig.show()


**¿Ayuda a predecir?** Sí, y mucho. Divido las filas en riesgo en cinco grupos del mismo tamaño según su carga de 14 días: el grupo con menos carga tiene un 0,56 % de lesiones nuevas y el de más carga un 1,54 %, casi el triple. El riesgo sube en cada grupo, sin excepciones, lo que apoya H1.

**¿Por qué 14 días?** He comparado tres ventanas (7, 14 y 28 días) y las tres dan casi el mismo resultado: el riesgo sube de forma continua desde alrededor del 0,55-0,59 % en el grupo con menos carga hasta alrededor del 1,47-1,54 % en el de más carga. Como ninguna ventana es claramente mejor, mantengo **14 días** porque es la ventana del semáforo del brief (el baseline que el modelo tiene que superar), y así modelo y semáforo se comparan con la misma medida.

**Precaución:** esta comparación no demuestra que la carga cause la lesión. Los jugadores con más carga suelen ser titulares y veteranos, y los partidos con más carga acumulada suelen ser de final de temporada. El modelo tendrá en cuenta todas las features a la vez, lo que permite separar mejor el efecto de cada una.

### Feature 2: minutos del último partido (`minutes_last_game`)

**Qué mide:** los minutos que el jugador jugó en su partido anterior. La carga de 14 días mide la fatiga acumulada; esta feature mide el último esfuerzo. Un partido muy largo justo antes podría dejar al jugador más expuesto que la misma carga repartida en varios partidos.

**Cómo la construyo:** con `shift(1)` dentro de cada jugador, que trae el valor de su fila anterior. No hace falta rellenar vacíos: las filas en riesgo exigen que el jugador jugara el partido anterior, así que siempre tienen minutos.

In [139]:
#PRIMERO: minutos del partido anterior de cada jugador
#shift(1) dentro de cada jugador trae el valor de su fila anterior (el partido ya jugado)
df["minutes_last_game"] = df.groupby("personId")["numMinutes"].shift(1)

#SEGUNDO: compruebo que en las filas en riesgo no hay vacíos (siempre jugaron el partido anterior)
print("Vacíos en filas en riesgo:", df.loc[df["at_risk"], "minutes_last_game"].isna().sum())

#TERCERO: comprobación de fuga con el mismo jugador: el valor de cada fila son los minutos de la fila ANTERIOR
check_player = df[(df["lastName"] == "Tatum") & (df["season_start"] == 2023)]
check_player[["game_day", "numMinutes", "minutes_last_game", "at_risk"]].head(5)

Vacíos en filas en riesgo: 0


,game_day,numMinutes,minutes_last_game,at_risk
209826,2023-10-25,38.0,41.0,False
209827,2023-10-27,41.0,38.0,True
209828,2023-10-30,27.0,41.0,True
209829,2023-11-01,26.0,27.0,True
209830,2023-11-04,40.0,26.0,True


**Comprobación de que no hay fuga de información:** en los partidos de Jayson Tatum, el valor de cada fila son los minutos de la fila anterior: el 27/10 vale 38 (los minutos del 25/10), el 30/10 vale 41 (los del 27/10), y así sucesivamente. Nunca usa los minutos del propio partido.

En su primer partido de la temporada (25/10) vale 41, que son los minutos de su último partido de la temporada anterior. No es un problema: esa fila no está en riesgo (`at_risk = False`) porque su partido anterior es de otra temporada, así que no entra en el modelo. No hay vacíos en las filas en riesgo, porque siempre jugaron el partido anterior.

In [140]:
#PRUEBA: ¿AYUDA A PREDECIR?
#Tasa de lesión según los minutos del último partido, en tramos con sentido de baloncesto
#Menos de 10 (minutos residuales), 10-20 (suplente), 20-30 (rotación), 30-36 (titular), 36 o más (partido muy largo)
at_risk_rows = df[df["at_risk"]]
minutes_band = pd.cut(
    at_risk_rows["minutes_last_game"], bins=[0, 10, 20, 30, 36, 100], right=False,
    labels=["<10", "10-20", "20-30", "30-36", "36+"],
)
last_game_rate = at_risk_rows.groupby(minutes_band, observed=True)["lesion_nueva"].agg(["mean", "size"]).reset_index()
last_game_rate.columns = ["minutes_band", "injury_rate", "rows"]
last_game_rate["injury_rate"] = last_game_rate["injury_rate"] * 100

fig = px.bar(
    last_game_rate, x="minutes_band", y="injury_rate", text="rows",
    title="Tasa de lesión nueva según los minutos del último partido",
    labels={"minutes_band": "Minutos jugados en el partido anterior", "injury_rate": "Lesiones nuevas (%)"},
)
fig.update_traces(texttemplate="%{text:,} filas")
fig.show()

#Relación con la carga de 14 días: si es muy alta, las dos features dicen casi lo mismo
print("Correlación con load_14d:", round(at_risk_rows[["minutes_last_game", "load_14d"]].corr().iloc[0, 1], 2))

Correlación con load_14d: 0.72


**¿Ayuda a predecir?** Un poco. La tasa de lesión sube con los minutos jugados en el partido anterior, del 0,80 % (menos de 10 minutos) al 1,25 % (entre 30 y 36), pero baja otra vez en los partidos muy largos (1,10 % con 36 minutos o más). Mi idea de que un partido muy largo justo antes aumenta el riesgo no se confirma: los jugadores que juegan tantos minutos suelen ser estrellas a las que el equipo cuida más.

Además, se parece bastante a la carga de 14 días (correlación de 0,72): quien juega muchos minutos en un partido suele jugarlos en todos.

**Decisión:** la mantengo de momento, porque un modelo de árboles puede encontrar combinaciones (por ejemplo, un partido largo seguido de poco descanso) que no se ven mirando la feature sola. En el apartado 9 comprobaré con la importancia por permutación si el modelo la usa; si no aporta, la quitaré.

### Feature 3: días desde el último partido (`days_since_last_game`)

**Qué mide:** cuántos días han pasado desde el último partido del jugador. Un 1 significa back-to-back (jugó ayer), un 2 significa un día de descanso entre partidos, y así sucesivamente. Es la otra cara de la carga: la carga mide cuánto ha jugado, y esta feature mide cuánto ha podido recuperarse.

**Decisiones:**
- **Usar la fecha del partido que predigo no es fuga de información.** El calendario se publica meses antes, así que la víspera sé cuándo se juega. Lo que no puedo usar es lo que pasa en el partido (minutos, puntos), pero sí cuándo se juega.
- **Limito el valor a 10 días.** Hay muy pocas filas con descansos más largos, y casi todas son de la vuelta a la competición en la burbuja de Orlando (verano de 2020), tras más de 140 días de parón por el COVID. No es un descanso normal, y un valor de 146 días distorsionaría la regresión logística. A partir de unos 8 días el riesgo ya no baja más, así que para el modelo 10 días y 146 días significan lo mismo: descanso completo.
- **No hay vacíos** en las filas en riesgo, porque el jugador siempre jugó el partido anterior de su equipo.

In [141]:
#PRIMERO: días desde el último partido de cada jugador
#diff() dentro de cada jugador resta la fecha de su partido anterior a la del actual (1 = back-to-back)
df["days_since_last_game"] = df.groupby("personId")["game_day"].diff().dt.days

#SEGUNDO: compruebo los descansos muy largos en las filas en riesgo, y de cuándo son
long_breaks = df[df["at_risk"] & (df["days_since_last_game"] > 10)]
print("Filas en riesgo con más de 10 días sin jugar:", len(long_breaks))
print(long_breaks["gameDate"].dt.to_period("M").value_counts().head(5))

#TERCERO: limito el valor a 10 días (más allá, el descanso ya es completo)
df["days_since_last_game"] = df["days_since_last_game"].clip(upper=10)

#CUARTO: compruebo que en las filas en riesgo no hay vacíos
print("\nVacíos en filas en riesgo:", df.loc[df["at_risk"], "days_since_last_game"].isna().sum())

#QUINTO: comprobación de fuga con el mismo jugador: los días se cuentan desde su partido ANTERIOR
check_player = df[(df["lastName"] == "Tatum") & (df["season_start"] == 2023)]
check_player[["game_day", "days_since_last_game", "at_risk"]].head(6)

Filas en riesgo con más de 10 días sin jugar: 217
gameDate
2020-07    132
2020-08     52
2021-01     17
2020-12     16
Freq: M, Name: count, dtype: int64

Vacíos en filas en riesgo: 0


,game_day,days_since_last_game,at_risk
209826,2023-10-25,10.0,False
209827,2023-10-27,2.0,True
209828,2023-10-30,3.0,True
209829,2023-11-01,2.0,True
209830,2023-11-04,3.0,True
209831,2023-11-06,2.0,True


**Comprobación:** hay 217 filas en riesgo con más de 10 días sin jugar, y casi todas son de julio (132) y agosto (52) de 2020: la vuelta a la competición en la burbuja de Orlando. Las de diciembre de 2020 (16) y enero de 2021 (17) vienen de una limitación de mi regla de temporadas: la final de 2019-20 se jugó en octubre de 2020 y la regla la cuenta como temporada 2020-21. Son muy pocas filas y el límite de 10 días las neutraliza.

En los partidos de Jayson Tatum, cada valor son los días desde su partido anterior (2, 3, 2…), así que no usa información del futuro. Su primer partido de la temporada vale 10 porque venía de la temporada anterior, pero no está en riesgo. No hay vacíos en las filas en riesgo.

In [142]:
#PRUEBA ¿AYUDA A PREDECIR?
#Tasa de lesión según los días desde el último partido
at_risk_rows = df[df["at_risk"]]
rest_band = pd.cut(
    at_risk_rows["days_since_last_game"], bins=[1, 2, 3, 4, 8, 11], right=False,
    labels=["1 (back-to-back)", "2", "3", "4-7", "8-10"],
)
rest_rate = at_risk_rows.groupby(rest_band, observed=True)["lesion_nueva"].agg(["mean", "size"]).reset_index()
rest_rate.columns = ["rest_band", "injury_rate", "rows"]
rest_rate["injury_rate"] = rest_rate["injury_rate"] * 100

fig = px.bar(
    rest_rate, x="rest_band", y="injury_rate", text="rows",
    title="Tasa de lesión nueva según los días desde el último partido",
    labels={"rest_band": "Días desde el último partido", "injury_rate": "Lesiones nuevas (%)"},
)
fig.update_traces(texttemplate="%{text:,} filas")
fig.show()

#Relación con la carga de 14 días: si es baja, aporta información distinta
print("Correlación con load_14d:", round(at_risk_rows[["days_since_last_game", "load_14d"]].corr().iloc[0, 1], 2))

Correlación con load_14d: -0.11


**¿Ayuda a predecir?** Sí, mucho. El riesgo baja de forma continua a medida que aumenta el descanso: un 1,80 % en un back-to-back, un 1,04 % con un día de descanso entre partidos y un 0,61 % con 8 días o más. Un jugador que juega dos días seguidos tiene casi el triple de riesgo que uno que ha descansado más de una semana.

Además, casi no se parece a la carga de 14 días (correlación de −0,11), así que aporta información distinta: dos jugadores con la misma carga pueden tener riesgos diferentes según cuánto hayan descansado justo antes.

**Para el cuerpo técnico:** los back-to-back son el momento de más riesgo, así que son el primer candidato para limitar minutos o dar descanso a los jugadores más cargados.

| Feature | How it is built | Why it should help | Known at prediction time? |
|---|---|---|---|
| *TODO* | | | yes / no → dropped |

### Feature 4: back-to-back (`back_to_back`)

**Qué mide:** si el jugador jugó el día anterior (1) o no (0). Sale directamente de la feature 3: es 1 cuando `days_since_last_game` vale 1.

**¿Por qué no basta con la feature 3?** En el gráfico de días de descanso, el salto entre jugar ayer y tener un día de descanso (de 1,80 % a 1,04 %, −0,76 puntos) es cuatro veces mayor que el siguiente (de 1,04 % a 0,86 %, −0,18 puntos). La regresión logística trata los días como una línea recta, con el mismo efecto por cada día de descanso, así que no puede captar ese salto. Con una columna aparte para el back-to-back sí puede. A los modelos de árboles no les molesta tenerla. Además, la necesito para comprobar H2 y para la regla de decisión de minutos.

In [143]:
#PRIMERO: back-to-back = el jugador jugó el día anterior (1 día desde su último partido)
df["back_to_back"] = (df["days_since_last_game"] == 1).astype(int)

#SEGUNDO: cuántos back-to-back hay y su tasa de lesión, en las filas en riesgo
at_risk_rows = df[df["at_risk"]]
print(f"Back-to-back: {at_risk_rows['back_to_back'].mean():.1%} de las filas en riesgo")

b2b_rate = at_risk_rows.groupby("back_to_back")["lesion_nueva"].agg(["size", "sum", "mean"])
b2b_rate["mean"] = (b2b_rate["mean"] * 100).round(2)
b2b_rate.index = ["No back-to-back", "Back-to-back"]
print(b2b_rate)

#TERCERO: relación con la carga de 14 días
print("Correlación con load_14d:", round(at_risk_rows[["back_to_back", "load_14d"]].corr().iloc[0, 1], 2))

Back-to-back: 17.0% de las filas en riesgo
                   size   sum  mean
No back-to-back  213939  2115  0.99
Back-to-back      43949   790   1.8
Correlación con load_14d: 0.04


**¿Ayuda a predecir?** Sí. El 17,0 % de los partidos en riesgo son back-to-back. En ellos hay 790 lesiones nuevas de 43.949 partidos (1,80 %), frente a 2.115 de 213.939 en el resto (0,99 %): casi el doble de riesgo. Además, casi no se parece a la carga de 14 días (correlación de 0,04), así que aporta información distinta: un jugador puede llegar a un back-to-back con mucha o con poca carga acumulada.

### Prueba de H2: ¿baja la eficiencia tras un back-to-back?

Mido la eficiencia con el **TS% (True Shooting)**, que calcula los puntos que consigue un jugador por cada intento de tiro, contando los tiros libres: TS% = puntos / (2 × (tiros de campo intentados + 0,44 × tiros libres intentados)) (Basketball Reference, s. f.).

**El TS% no es una feature:** se calcula con los tiros del propio partido, así que solo se conoce después de jugarlo. Usarlo en el modelo sería fuga de información. Lo calculo solo para comprobar H2.

Uso los partidos en los que el jugador jugó y también había jugado el partido anterior de su equipo. No exijo comentarios, porque el TS% no los necesita, así que aquí sí entra la temporada 2021-22. Hago dos comparaciones, porque una sola puede engañar:
1. **Global:** el TS% de todos los partidos tras un back-to-back frente al de los demás. Sumo puntos y tiros de todos los partidos, para que no pese igual un jugador con 2 tiros que uno con 20. El riesgo de esta comparación es que, si en los back-to-back juegan más los suplentes, el TS% bajaría sin que nadie estuviera más cansado.
2. **Jugador a jugador:** cada jugador comparado consigo mismo, para evitar ese problema. Solo uso jugadores con al menos 100 tiros (contando los libres) en partidos con descanso y 50 en back-to-back, para que su TS% no dependa de unos pocos tiros. En back-to-back pido menos porque hay muchos menos partidos de ese tipo.

Para saber si la diferencia jugador a jugador es real o puede ser casualidad, uso una **prueba t emparejada**: si el p-valor es menor de 0,05, la diferencia se considera real.

In [144]:
from scipy import stats

#PRIMERO: partidos en los que el jugador jugó y también jugó el partido anterior de su equipo (no exijo comentarios)
h2_games = df[
    (df["motivo_ausencia"] == "jugó") & played_prev_game & same_season & played_team_prev_game
].copy()
#Intentos de tiro "equivalentes": tiros de campo + 0,44 × tiros libres
h2_games["shot_attempts"] = h2_games["fieldGoalsAttempted"] + 0.44 * h2_games["freeThrowsAttempted"]

#SEGUNDO: comparación global (sumo puntos y tiros de todos los partidos, para que no pesen igual un jugador con 2 tiros que uno con 20)
h2_global = h2_games.groupby("back_to_back").agg(
    games=("points", "size"), points=("points", "sum"), shot_attempts=("shot_attempts", "sum"), avg_minutes=("numMinutes", "mean")
)
h2_global["ts_pct"] = (h2_global["points"] / (2 * h2_global["shot_attempts"]) * 100).round(2)
h2_global.index = ["Con descanso", "Back-to-back"]
print(h2_global[["games", "avg_minutes", "ts_pct"]])

#TERCERO: comparación jugador a jugador (cada jugador consigo mismo)
by_player_b2b = h2_games.groupby(["personId", "back_to_back"])[["points", "shot_attempts"]].sum().unstack()
#Solo jugadores con al menos 100 tiros con descanso y 50 en back-to-back, para que su TS% sea estable
enough_shots = (by_player_b2b[("shot_attempts", 0)] >= 100) & (by_player_b2b[("shot_attempts", 1)] >= 50)
by_player_b2b = by_player_b2b[enough_shots]
ts_rested = by_player_b2b[("points", 0)] / (2 * by_player_b2b[("shot_attempts", 0)]) * 100
ts_b2b = by_player_b2b[("points", 1)] / (2 * by_player_b2b[("shot_attempts", 1)]) * 100
ts_change = ts_b2b - ts_rested

print(f"\nJugadores comparados: {len(ts_change)}")
print(f"Cambio medio del TS% en back-to-back: {ts_change.mean():+.2f} puntos")
print(f"Jugadores que tiran peor en back-to-back: {(ts_change < 0).mean():.0%}")

#CUARTO: ¿la diferencia es real o puede ser casualidad? (prueba t para datos emparejados)
#Si el p-valor es mayor que 0,05, no puedo descartar que la diferencia sea casualidad
t_test = stats.ttest_rel(ts_b2b, ts_rested)
print(f"p-valor: {t_test.pvalue:.2f}")

               games  avg_minutes  ts_pct
Con descanso  215050    23.681989   56.81
Back-to-back   43579    23.743312   56.31

Jugadores comparados: 884
Cambio medio del TS% en back-to-back: -0.22 puntos
Jugadores que tiran peor en back-to-back: 53%
p-valor: 0.16


**Resultado de H2:** la eficiencia apenas cambia tras un back-to-back.
- **Global:** el TS% baja de 56,81 % (215.050 partidos con descanso) a 56,31 % (43.579 back-to-back), medio punto. Los minutos de media son los mismos (23,7), así que no es que en los back-to-back jueguen menos.
- **Jugador a jugador:** en los 884 jugadores comparados consigo mismos, el cambio medio es de −0,22 puntos, y solo el 53 % tira peor en back-to-back: casi la mitad tira mejor.
- **Prueba t emparejada:** p-valor de 0,16, mayor que 0,05, así que no puedo descartar que esta pequeña diferencia sea casualidad.

**H2 no se confirma.** El back-to-back no reduce de forma clara la eficiencia, pero sí casi duplica el riesgo de lesión nueva (1,80 % frente a 0,99 %, feature 4). Para el cuerpo técnico, la conclusión es que descansar a un jugador en un back-to-back sirve para reducir el riesgo de lesión, no para que rinda mejor.

### Feature 5: titular habitual (`starter_share`)

**Qué mide:** el porcentaje de partidos como titular en los 10 últimos partidos que ha jugado el jugador (1 = siempre titular, 0 = siempre suplente). Sale de `startingPosition`, que solo tiene valor (G, F o C) cuando el jugador sale de titular.

**Por qué debería ayudar:** los titulares no solo juegan más minutos, sino los más intensos (inicios de partido y finales igualados) y contra los mejores rivales. Es una medida del rol del jugador que la carga de minutos no recoge del todo.

**Decisiones:**
- **Porcentaje, no "sí o no".** Si marcara "titular habitual" a partir de un corte (por ejemplo, más del 50 %), perdería información: no es lo mismo un 60 % que un 100 %. Con el porcentaje, el modelo decide dónde está el corte.
- **Una ventana de 10 partidos, no de días.** Aquí no mido fatiga sino rol, y el rol se ve en los partidos jugados, no en el calendario. Diez partidos son unas tres semanas: suficiente para que un partido suelto como titular no cambie mucho el valor, y lo bastante corto para recoger un cambio de rol (por ejemplo, un suplente que pasa a titular por la lesión de un compañero).
- **Solo cuento los partidos jugados.** Si contara también las ausencias como "no titular", un titular que vuelve de una lesión parecería suplente, y la feature mezclaría el rol con el historial de lesiones, que mide la feature 6. Lo compruebo con datos más abajo.
- **Sin fuga de información.** Calculo el porcentaje de cada partido jugado incluyéndolo, y después uso `shift(1)`, así que la feature de cada fila es el valor de su partido anterior: si el jugador es titular en el partido que predigo (algo que solo se sabe al empezar) no entra en la feature.
- **Al principio de cada temporada**, la ventana incluye los últimos partidos de la temporada anterior. Es correcto: el rol con el que acabó la temporada pasada es la mejor información que hay la víspera del primer partido.

In [145]:
#PRIMERO: ¿fue titular en cada partido? (startingPosition tiene valor, G, F o C, solo si salió de titular)
df["is_starter"] = df["startingPosition"].notna().astype(int)

#SEGUNDO: en cada partido JUGADO, % de titularidades en sus 10 últimos partidos jugados (incluido ese)
#Solo cuento partidos jugados, para que las ausencias por lesión no hagan parecer suplente a un titular
played_rows = df["motivo_ausencia"] == "jugó"
df["starter_share_incl"] = np.nan
df.loc[played_rows, "starter_share_incl"] = (
    df[played_rows]
    .groupby("personId")["is_starter"]
    .transform(lambda s: s.rolling(10, min_periods=1).mean())
)

#TERCERO: la feature es el valor de su partido ANTERIOR (shift(1)), así no incluye si es titular en el partido que predigo
df["starter_share"] = df.groupby("personId")["starter_share_incl"].shift(1)

#CUARTO: compruebo que no hay vacíos en las filas en riesgo y miro cómo se reparte
at_risk_rows = df[df["at_risk"]]
print("Vacíos en filas en riesgo:", at_risk_rows["starter_share"].isna().sum())
print(at_risk_rows["starter_share"].describe())

#QUINTO: comprobación de fuga con un jugador que cambia de rol: Al Horford (Celtics), inicio de 2023-24
check_player = df[(df["lastName"] == "Horford") & (df["season_start"] == 2023)]
check_player[["game_day", "motivo_ausencia", "is_starter", "starter_share"]].head(10)

Vacíos en filas en riesgo: 0
count    257888.000000
mean          0.559051
std           0.441365
min           0.000000
25%           0.000000
50%           0.700000
75%           1.000000
max           1.000000
Name: starter_share, dtype: float64


,game_day,motivo_ausencia,is_starter,starter_share
27154,2023-10-25,jugó,0,1.0
27155,2023-10-27,jugó,0,0.9
27156,2023-10-30,jugó,0,0.8
27157,2023-11-01,jugó,0,0.7
27158,2023-11-04,jugó,1,0.6
27159,2023-11-06,jugó,1,0.6
27160,2023-11-08,jugó,0,0.6
27161,2023-11-11,jugó,0,0.5
27162,2023-11-13,jugó,0,0.4
27163,2023-11-15,jugó,1,0.3


**Resultado:** en las filas en riesgo no hay vacíos. La media es 0,56 y la mediana 0,70: algo más de la mitad de las filas son de jugadores que salen de titulares más de la mitad de las veces. Es lógico, porque los titulares juegan más partidos y casi nunca les falta el anterior, así que tienen más filas en riesgo.

**Comprobación de que no hay fuga de información:** Al Horford empezó la temporada 2023-24 saliendo desde el banquillo. El 25/10 su valor es 1,0 (sus 10 últimos partidos jugados de la temporada anterior fueron como titular), y baja 0,1 en cada partido como suplente: 0,9, 0,8, 0,7 y 0,6. El 04/11 sale de titular, pero su valor ese día (0,6) no incluye esa titularidad: la recoge el partido siguiente. El 06/11 sigue en 0,6 porque entra la titularidad del 04/11 y sale de la ventana una de la temporada anterior.

In [146]:
#COMPROBACION DE LAS DOS OPCIONES Y LA PRUEBA DE AYUDA A PREDECIR
# #PRIMERO: opción A, solo para comparar: los 10 partidos anteriores del jugador, contando las ausencias como "no titular"
share_all_games = df.groupby("personId")["is_starter"].transform(lambda s: s.shift(1).rolling(10, min_periods=1).mean())

#SEGUNDO: tasa de lesión por tramos de % de titularidades, con las dos opciones
share_bands = {"0 %": (-0.01, 0), "1-50 %": (0, 0.5), "51-99 %": (0.5, 0.99), "100 %": (0.99, 1)}
band_edges = [-0.01, 0, 0.5, 0.99, 1]
band_labels = list(share_bands.keys())

comparison = pd.DataFrame({
    "A: todos los partidos (%)": at_risk_rows["lesion_nueva"].groupby(
        pd.cut(share_all_games[at_risk_rows.index], band_edges, labels=band_labels), observed=True).mean() * 100,
    "B: solo partidos jugados (%)": at_risk_rows["lesion_nueva"].groupby(
        pd.cut(at_risk_rows["starter_share"], band_edges, labels=band_labels), observed=True).mean() * 100,
})
print(comparison.round(2))

#TERCERO: gráfico de la opción elegida (B)
starter_rate = at_risk_rows.groupby(
    pd.cut(at_risk_rows["starter_share"], band_edges, labels=band_labels), observed=True
)["lesion_nueva"].agg(["mean", "size"]).reset_index()
starter_rate.columns = ["starter_band", "injury_rate", "rows"]
starter_rate["injury_rate"] = starter_rate["injury_rate"] * 100

fig = px.bar(
    starter_rate, x="starter_band", y="injury_rate", text="rows",
        title="Tasa de lesión nueva según el % de titularidades",
    labels={"starter_band": "Partidos como titular (de los 10 últimos jugados)", "injury_rate": "Lesiones nuevas (%)"},
)
fig.update_traces(texttemplate="%{text:,} filas")
fig.show()

#CUARTO: relación con la carga de 14 días (los titulares juegan más minutos)
print("Correlación con load_14d:", round(at_risk_rows[["starter_share", "load_14d"]].corr().iloc[0, 1], 2))

         A: todos los partidos (%)  B: solo partidos jugados (%)
0 %                           0.67                          0.66
1-50 %                        1.15                          1.06
51-99 %                       1.84                          1.31
100 %                         1.16                           1.4


Correlación con load_14d: 0.56


**¿Por qué cuento solo los partidos jugados?** Comparo las dos formas de calcular la feature:

| % de titularidades | 0 % | 1-50 % | 51-99 % | 100 % |
|---|---|---|---|---|
| A: todos los partidos (las ausencias cuentan como "no titular") | 0,67 % | 1,15 % | **1,84 %** | 1,16 % |
| B: solo partidos jugados (la elegida) | 0,66 % | 1,06 % | 1,31 % | 1,40 % |

Con la opción A, el grupo de 51-99 % tiene muchas más lesiones que los titulares fijos (1,84 % frente a 1,16 %). Es lo que esperaba: ahí caen los titulares que vuelven de una baja, porque sus partidos de ausencia cuentan como "no titular". Y quien acaba de estar lesionado tiene más riesgo de recaer. Ese efecto es real, pero lo mide la feature 6 (lesiones previas). Si lo mezclara aquí, las dos features se pisarían y no sabría cuál de las dos explica el riesgo. Con la opción B, la feature mide solo el rol.

**¿Ayuda a predecir?** Sí. Con la opción B, el riesgo sube con el rol de titular: del 0,66 % en los jugadores que nunca han salido de titular en sus 10 últimos partidos al 1,40 % en los que siempre lo han hecho, más del doble.

Se parece moderadamente a la carga de 14 días (correlación de 0,56), porque los titulares juegan más minutos, pero no es lo mismo: con la misma carga, un titular juega los minutos más intensos. El modelo verá las dos a la vez, y en el apartado 9 comprobaré cuánto aporta cada una.

### Feature 6: lesiones previas en la temporada (`prior_injuries_season`)

**Qué mide:** cuántas lesiones ha tenido ya el jugador en esa temporada antes del partido que se predice.

**Por qué debería ayudar:** quien ya se ha lesionado tiene más riesgo de volver a lesionarse, porque puede arrastrar una zona debilitada, volver sin estar al 100 % o tener una dolencia crónica.

**Decisiones:**
- **Cuento episodios, no filas.** Una lesión que dura 10 partidos tiene 10 filas "lesion", pero es una sola lesión. Un episodio empieza en una fila "lesion" cuyo partido anterior no fue "lesion".
- **Cuento todos los episodios, no solo los del target.** El target solo cuenta como nueva la lesión de un jugador que jugó el partido anterior de su equipo, porque así lo exige la pregunta del modelo. Pero para el historial del jugador importan todas sus lesiones, también las que empezaron justo después de un descanso.
- **Solo la temporada actual.** Es la definición del brief, y además evita un problema: como 2021-22 no tiene comentarios, no sé qué lesiones hubo. Si contara las lesiones de toda la carrera, ese hueco arrastraría un error a las temporadas siguientes; reiniciando cada temporada, no se propaga.
- **Sin fuga de información.** En la fila donde el jugador se lesiona, su propia lesión es el inicio de un episodio. Si la contara, la feature diría "ya lleva una lesión" justo en esa fila, y el modelo aprendería a predecirla con la respuesta delante. Por eso acumulo los episodios y resto el de la propia fila: solo cuentan los anteriores.

In [147]:
#PRIMERO: marco el inicio de cada episodio de lesión: una fila "lesion" cuyo partido anterior NO fue "lesion"
#Cuento todos los episodios (también los que empiezan tras un descanso), no solo los del target,
#porque para el historial del jugador importan todas sus lesiones
is_injury_row = df["motivo_ausencia"] == "lesion"
df["injury_start"] = (is_injury_row & (df["prev_status"] != "lesion")).astype(int)
print("Filas 'lesion':", is_injury_row.sum())
print("Episodios de lesión:", df["injury_start"].sum())
print("Lesiones nuevas del target:", int(df["lesion_nueva"].sum()))

#SEGUNDO: lesiones previas en la temporada = episodios acumulados en la temporada ANTES de este partido
#cumsum() va sumando dentro de cada jugador y temporada; resto el de la propia fila
#para no contar la lesión del partido que quiero predecir (fuga de información)
df["prior_injuries_season"] = (
    df.groupby(["personId", "season_start"])["injury_start"].cumsum() - df["injury_start"]
)

#TERCERO: cómo se reparte en las filas en riesgo
at_risk_rows = df[df["at_risk"]]
print("\nVacíos en filas en riesgo:", at_risk_rows["prior_injuries_season"].isna().sum())
print(at_risk_rows["prior_injuries_season"].value_counts().sort_index().head(6))
print("Máximo:", at_risk_rows["prior_injuries_season"].max())

#CUARTO: comprobación de fuga con Al Horford: su primera lesión de 2023-24 y los partidos de alrededor
check_player = df[(df["lastName"] == "Horford") & (df["season_start"] == 2023)].reset_index(drop=True)
first_injury = check_player.index[check_player["injury_start"] == 1][0]
check_player.loc[first_injury - 2 : first_injury + 3,
                 ["game_day", "motivo_ausencia", "injury_start", "lesion_nueva", "prior_injuries_season"]]

Filas 'lesion': 8113
Episodios de lesión: 4063
Lesiones nuevas del target: 2905

Vacíos en filas en riesgo: 0
prior_injuries_season
0    195370
1     41902
2     13252
3      4623
4      1840
5       583
Name: count, dtype: int64
Máximo: 17


,game_day,motivo_ausencia,injury_start,lesion_nueva,prior_injuries_season
57,2024-03-23,jugó,0,<NA>,0
58,2024-03-25,jugó,0,0,0
59,2024-03-28,lesion,1,1,0
60,2024-03-30,jugó,0,<NA>,1
61,2024-04-01,jugó,0,0,1
62,2024-04-03,jugó,0,0,1


**Resultado:** las 8.113 filas "lesion" forman 4.063 episodios, más que las 2.905 lesiones nuevas del target, porque aquí también cuentan las lesiones que empiezan tras un descanso o tras no jugar. No hay vacíos en las filas en riesgo. La mayoría (195.370) son de jugadores sin lesiones previas en la temporada; 41.902 tienen una, 13.252 dos, y a partir de ahí cada vez menos. El máximo es 17 (lo explico más abajo).

**Comprobación de que no hay fuga de información:** Al Horford se lesiona el 28/03/2024 (`lesion_nueva` = 1). Ese día su `prior_injuries_season` vale 0: su propia lesión no cuenta. A partir del partido siguiente (30/03) vale 1.

In [148]:
#PRUEBA: ¿AYUDA A PREDECIR? Y EL CASO MÁXIMO DE LESIONES PREVIAS
#PRIMERO: tasa de lesión según las lesiones previas en la temporada (0, 1, 2, 3 o más)
prior_band = pd.cut(at_risk_rows["prior_injuries_season"], bins=[-1, 0, 1, 2, 100], labels=["0", "1", "2", "3 o más"])
prior_rate = at_risk_rows.groupby(prior_band, observed=True)["lesion_nueva"].agg(["mean", "size"]).reset_index()
prior_rate.columns = ["prior_band", "injury_rate", "rows"]
prior_rate["injury_rate"] = prior_rate["injury_rate"] * 100

fig = px.bar(
    prior_rate, x="prior_band", y="injury_rate", text="rows",
    title="Tasa de lesión nueva según las lesiones previas en la temporada",
    labels={"prior_band": "Lesiones anteriores en la temporada", "injury_rate": "Lesiones nuevas (%)"},
)
fig.update_traces(texttemplate="%{text:,} filas")
fig.show()

#SEGUNDO: relación con la carga de 14 días
print("Correlación con load_14d:", round(at_risk_rows[["prior_injuries_season", "load_14d"]].corr().iloc[0, 1], 2))

#TERCERO: ¿quién tiene el máximo de lesiones previas, y cómo son sus lesiones?
max_row = at_risk_rows["prior_injuries_season"].idxmax()
max_player = df.loc[max_row, ["personId", "season_start", "firstName", "lastName"]]
print(f"\nMáximo: {max_player['firstName']} {max_player['lastName']}, temporada {max_player['season_start']}")
max_player_season = df[(df["personId"] == max_player["personId"]) & (df["season_start"] == max_player["season_start"])]
print(max_player_season["motivo_ausencia"].value_counts())
max_player_season.loc[max_player_season["injury_start"] == 1, ["game_day", "comment"]].head(10)

Correlación con load_14d: 0.05

Máximo: Robert Williams III, temporada 2025
motivo_ausencia
jugó      65
lesion    18
otros      3
Name: count, dtype: int64


,game_day,comment
256889,2025-11-19,DND - Injury/Illness
256891,2025-11-23,DND - Injury/Illness
256896,2025-12-03,DND - Injury/Illness
256898,2025-12-07,DND - Injury/Illness
256904,2025-12-23,DND - Injury/Illness
256906,2025-12-28,DND - Injury/Illness
256910,2026-01-03,DND - Injury/Illness
256918,2026-01-18,DNP - Injury/Illness
256920,2026-01-23,DND - Injury/Illness
256922,2026-01-27,DND - Injury/Illness


**El valor máximo (17) no es un error.** Es Robert Williams III en la temporada 2025-26: jugó 65 partidos y tiene 18 filas "lesion". Como llega a 17 lesiones previas, cada una de esas 18 filas es un episodio distinto, es decir, cada lesión duró un solo partido: se pierde un partido, vuelve a jugar y, unas semanas después, se repite. Los comentarios dicen "____". Parece una dolencia que el equipo gestiona dándole descanso en partidos sueltos. Es justo el tipo de jugador que esta feature tiene que detectar: alguien que se resiente con frecuencia.

**Limitación:** en casos así, cada partido de descanso por la misma dolencia cuenta como una lesión nueva, tanto en el target como en esta feature. No puedo distinguir una lesión nueva de una molestia crónica que reaparece, porque la mayoría de comentarios solo dicen "Injury/Illness".

### Feature 7: fase final (`is_playoffs`)

**Qué mide:** si el partido es de fase final (1) o no (0).

**Por qué debería ayudar:** en el EDA (apartado 3) vi que la tasa de lesión nueva en playoffs es mucho más baja que en temporada regular. Sin esta feature, el modelo aplicaría a un partido de playoffs el mismo riesgo que a uno de temporada regular, y daría demasiadas alertas justo en los partidos más importantes. No es fuga de información: el tipo de partido se sabe de antemano.

**Decisión: qué partidos cuentan como fase final.** Antes de decidir, comparo la tasa de lesión de cada tipo de partido junto con el número de lesiones, porque con pocos casos el porcentaje no es fiable. Tengo tres opciones:
- **Una columna por tipo de partido:** el modelo aprendería un efecto para la NBA Cup y otro para el Play-in con muy pocas lesiones, lo que sería ruido.
- **"No es temporada regular"** (Playoffs, Play-in y NBA Cup): mezclaría la NBA Cup, que se juega en mitad de la temporada regular, con los partidos de fase final.
- **Fase final = Playoffs y Play-in (la elegida):** agrupa los partidos que se juegan después de la temporada regular y son eliminatorios. La NBA Cup queda con la temporada regular, porque se juega en el mismo calendario.

In [149]:
#PRIMERO: tasa de lesión por tipo de partido, con el número de lesiones (no solo el %)
#Con pocas lesiones, el porcentaje no es fiable
at_risk_rows = df[df["at_risk"]]
game_type_rate = at_risk_rows.groupby("gameType")["lesion_nueva"].agg(["size", "sum", "mean"])
game_type_rate["mean"] = (game_type_rate["mean"] * 100).round(2)
game_type_rate.columns = ["Filas en riesgo", "Lesiones nuevas", "Tasa (%)"]
print(game_type_rate.sort_values("Filas en riesgo", ascending=False))

#SEGUNDO: la feature: 1 si es un partido de fase final (Playoffs o Play-in), 0 si no
#La NBA Cup queda con la temporada regular: se juega en el mismo calendario y solo tiene 3 lesiones, demasiado pocas para decidir
df["is_playoffs"] = df["gameType"].isin(["Playoffs", "Play-in"]).astype(int)

#TERCERO: tasa de lesión con la feature ya creada
at_risk_rows = df[df["at_risk"]]
playoff_rate = at_risk_rows.groupby("is_playoffs")["lesion_nueva"].agg(["size", "sum", "mean"])
playoff_rate["mean"] = (playoff_rate["mean"] * 100).round(2)
playoff_rate.index = ["Temporada regular (con NBA Cup)", "Fase final (Playoffs y Play-in)"]
playoff_rate.columns = ["Filas en riesgo", "Lesiones nuevas", "Tasa (%)"]
print("\n", playoff_rate)

#CUARTO: en fase final, ¿cambian la carga y el descanso?
print("\nCarga media de 14 días:")
print(at_risk_rows.groupby("is_playoffs")["load_14d"].mean().round(1))
print("Correlación con load_14d:", round(at_risk_rows[["is_playoffs", "load_14d"]].corr().iloc[0, 1], 2))
print("Correlación con days_since_last_game:", round(at_risk_rows[["is_playoffs", "days_since_last_game"]].corr().iloc[0, 1], 2))

                Filas en riesgo  Lesiones nuevas  Tasa (%)
gameType                                                  
Regular Season           238661             2852       1.2
Playoffs                  17220               49      0.28
NBA Cup                    1384                3      0.22
Play-in                     623                1      0.16

                                  Filas en riesgo  Lesiones nuevas  Tasa (%)
Temporada regular (con NBA Cup)           240045             2855      1.19
Fase final (Playoffs y Play-in)            17843               50      0.28

Carga media de 14 días:
is_playoffs
0    125.2
1    105.7
Name: load_14d, dtype: float64
Correlación con load_14d: -0.07
Correlación con days_since_last_game: 0.15


**Resultado:** la NBA Cup (0,22 %) y el Play-in (0,16 %) tienen tasas bajas, pero con solo 3 y 1 lesiones nuevas: una lesión más o menos cambiaría mucho el porcentaje, así que no son fiables. Por eso no baso la decisión en ellas y uso la agrupación por calendario: el Play-in va con los playoffs y la NBA Cup con la temporada regular.

**¿Ayuda a predecir?** Sí. En temporada regular (con la NBA Cup) hay 2.855 lesiones nuevas de 240.045 partidos (1,19 %); en fase final, 50 de 17.843 (0,28 %), cuatro veces menos.

**¿Lo explican ya la carga y el descanso?** Solo en parte. En fase final la carga media de 14 días es algo menor (105,7 minutos, frente a 125,2) y hay algo más de descanso entre partidos (correlación de 0,15 con los días desde el último partido), pero son diferencias moderadas, mientras que la de lesiones es muy grande. Las correlaciones con la carga (−0,07) y con el descanso (0,15) son bajas, así que la feature aporta información propia. Las causas pueden ser varias y no las puedo separar con estos datos: en playoffs los jugadores juegan con molestias que en temporada regular les harían descansar, y solo juegan los mejores equipos.

**Limitación:** con 3 lesiones no puedo saber si la NBA Cup tiene de verdad menos riesgo. Y el modelo aprende la fase final a partir de solo 50 lesiones nuevas, así que sus predicciones en playoffs serán menos fiables que en temporada regular.

### Edad (`edad`)

**Qué mide:** la edad del jugador el día del partido. Ya la calculé en la limpieza y en el apartado 3 vi que el riesgo sube con la edad (0,97 % en los menores de 25, 1,35 % en los de 30 o más), así que la incluyo como feature. No es fuga de información: la fecha de nacimiento y la del partido se conocen de antemano.

**Qué compruebo:** en la limpieza quedaron 11 filas sin fecha de nacimiento. Miro si alguna está en las filas en riesgo, que son las que usa el modelo.

In [150]:
#La edad ya existe (la calculé en la limpieza, el día de cada partido)
#En la limpieza quedaron 11 filas sin fecha de nacimiento: compruebo si alguna está en las filas en riesgo
at_risk_rows = df[df["at_risk"]]
print("Vacíos de edad en filas en riesgo:", at_risk_rows["edad"].isna().sum())
print("Jugadores sin edad (en todo el dataset):")
print(df.loc[df["edad"].isna(), ["firstName", "lastName", "at_risk"]].drop_duplicates(["firstName", "lastName"]))

print("\n", at_risk_rows["edad"].describe().round(1))
print("Correlación con load_14d:", round(at_risk_rows[["edad", "load_14d"]].corr().iloc[0, 1], 2))

Vacíos de edad en filas en riesgo: 0
Jugadores sin edad (en todo el dataset):
          firstName  lastName  at_risk
155360      Richard   Solomon    False
178462  Sir'Dominic   Pointer    False
229119         Mike     Young    False
231250       Yakuba  Ouattara    False
231941         Cody     Demps    False
255310       Keenan     Evans    False

 count    257888.0
mean         26.8
std           4.2
min          18.8
25%          23.5
50%          26.2
75%          29.6
max          43.1
Name: edad, dtype: float64
Correlación con load_14d: 0.04


**Resultado:** no hay ninguna edad vacía en las filas en riesgo. Las 11 filas sin fecha de nacimiento son de 6 jugadores (Richard Solomon, Sir'Dominic Pointer, Mike Young, Yakuba Ouattara, Cody Demps y Keenan Evans), y ninguna está en riesgo. Aun así, en el pipeline del modelo dejaré un paso que rellena la edad con la mediana, por si en el futuro llega un jugador sin fecha de nacimiento: es una protección que no cuesta nada.

En las filas en riesgo, la edad va de 18,8 a 43,1 años, con una mediana de 26,2. Casi no se parece a la carga de 14 días (correlación de 0,04), así que aporta información distinta.

### Features que entran en el modelo

Decido qué columnas puede ver el modelo. Es la decisión más importante contra la fuga de información, porque el dataset tiene muchas columnas que el modelo no puede usar.

**Entran 8 features**, todas numéricas: las 7 que he construido y la edad. Las de sí o no (`back_to_back` e `is_playoffs`) ya son 0 y 1, así que el modelo las usa como números sin transformarlas, y no hay ninguna categoría de texto que convertir (`CATEGORICAL` queda vacía).

**No entran:**

| Columnas | Por qué |
|---|---|
| `numMinutes`, `points`, `fieldGoalsAttempted`, `freeThrowsAttempted`, `is_starter`, `startingPosition` | **Fuga de información:** son del propio partido, que aún no se ha jugado |
| `comment`, `motivo_ausencia`, `injury_start` | **Fuga de información:** dicen si el jugador se lesionó; son la respuesta |
| `grupo_edad` | Repite la edad en tramos. Solo lo uso para comparar errores por grupos (apartado 10) |
| `gameType` | Ya está resumido en `is_playoffs` |
| `personId` y el nombre del jugador | El modelo memorizaría jugadores ("este se lesiona mucho") en lugar de aprender patrones de carga, y no serviría para un jugador nuevo |
| `playerteamName`, `opponentteamName` | El modelo aprendería equipos en lugar de patrones. Además, lo evalúo en los Celtics: si aprendiera cuánto se lesionan los Celtics, la evaluación sería engañosa. Y cada equipo puede registrar las lesiones de forma distinta |
| `season_start` | El modelo aprendería el nivel de lesiones de cada temporada, y las temporadas del test no las ha visto nunca. Además, la tasa cambia mucho entre temporadas (apartado 3) |
| `home` | No la planteé en el brief y no la he explorado. La dejo como posible mejora |
| `at_risk`, `prev_status`, `team_prev_game` y otras auxiliares | Las usé para construir el target y las features |

In [151]:
#Las listas que usará el pipeline del modelo
TARGET = "lesion_nueva"

#Las 8 features: todas son números (las de sí/no ya son 0 y 1, así que no hace falta transformarlas)
NUMERIC = [
    "load_14d",                #feature 1: minutos en los 14 días anteriores
    "minutes_last_game",       #feature 2: minutos del partido anterior
    "days_since_last_game",    #feature 3: días desde el último partido (máximo 10)
    "back_to_back",            #feature 4: jugó ayer (1) o no (0)
    "starter_share",           #feature 5: % de titularidades en los 10 últimos partidos jugados
    "prior_injuries_season",   #feature 6: lesiones anteriores en la temporada
    "is_playoffs",             #feature 7: fase final (1) o no (0)
    "edad",                    #edad el día del partido
]
#No hay categorías de texto que transformar
CATEGORICAL = []

#PRIMERO: compruebo que en las filas del modelo no falta ningún valor
model_rows = df[df["at_risk"]]
print("Vacíos por feature en las filas del modelo:")
print(model_rows[NUMERIC].isna().sum())

#SEGUNDO: resumen de las 8 features
print("\n", model_rows[NUMERIC].describe().T[["mean", "min", "max"]].round(2))

#TERCERO: correlaciones entre features
#Si dos features estuvieran muy correlacionadas (más de 0,8), dirían casi lo mismo y una sobraría
model_rows[NUMERIC].corr().round(2)

Vacíos por feature en las filas del modelo:
load_14d                 0
minutes_last_game        0
days_since_last_game     0
back_to_back             0
starter_share            0
prior_injuries_season    0
is_playoffs              0
edad                     0
dtype: int64

                          mean    min     max
load_14d               123.89   0.00  338.63
minutes_last_game       22.56   0.01   64.00
days_since_last_game     2.16   1.00   10.00
back_to_back             0.17   0.00    1.00
starter_share            0.56   0.00    1.00
prior_injuries_season    0.37   0.00   17.00
is_playoffs              0.07   0.00    1.00
edad                    26.76  18.82   43.12


,load_14d,minutes_last_game,days_since_last_game,back_to_back,starter_share,prior_injuries_season,is_playoffs,edad
load_14d,1.00,0.72,-0.11,0.04,0.56,0.05,-0.07,0.04
minutes_last_game,0.72,1.00,0.01,-0.01,0.57,0.10,-0.00,0.06
days_since_last_game,-0.11,0.01,1.00,-0.50,-0.00,0.03,0.15,0.02
back_to_back,0.04,-0.01,-0.50,1.00,0.00,-0.02,-0.12,-0.01
starter_share,0.56,0.57,-0.00,0.00,1.00,0.09,0.01,0.10
prior_injuries_season,0.05,0.10,0.03,-0.02,0.09,1.00,0.16,0.09
is_playoffs,-0.07,-0.00,0.15,-0.12,0.01,0.16,1.00,0.07
edad,0.04,0.06,0.02,-0.01,0.10,0.09,0.07,1.00


**Resultado:** ninguna de las 8 features tiene valores vacíos en las 257.888 filas del modelo.

**Correlaciones entre features:** ninguna pasa de 0,8, el valor a partir del cual dos features dirían casi lo mismo y una sobraría. Las más altas son:
- **Carga de 14 días y minutos del último partido (0,72):** quien juega muchos minutos en un partido suele jugarlos en todos. Es la más alta, pero aún por debajo de 0,8, y en el apartado 9 veré si los minutos del último partido aportan algo.
- **Titularidades con minutos del último partido (0,57) y con carga (0,56):** los titulares juegan más minutos.
- **Días desde el último partido y back-to-back (−0,50):** es lógico, porque el back-to-back sale de los días de descanso. No es una correlación muy alta, porque la mayoría de partidos no son back-to-back y tienen 2, 3 o más días de descanso.

Las demás están por debajo de 0,2: la edad, las lesiones previas, la fase final y el descanso aportan información distinta de la carga.

| Feature | How it is built | Why it should help | Known at prediction time? |
|---|---|---|---|
| `load_14d` | Suma de minutos en los 14 días naturales anteriores al partido (ventana que no incluye el propio partido) | H1: más carga, más riesgo. De 0,56 % (grupo con menos carga) a 1,54 % (con más) | **Sí:** son partidos ya jugados |
| `minutes_last_game` | Minutos del partido anterior (`shift(1)`) | El último esfuerzo. Efecto débil (de 0,80 % a 1,25 %, y baja a 1,10 % con 36 minutos o más) | **Sí:** partido ya jugado |
| `days_since_last_game` | Días desde el último partido, con un máximo de 10 | Recuperación: de 1,80 % (back-to-back) a 0,61 % (8 días o más) | **Sí:** el calendario se publica meses antes |
| `back_to_back` | 1 si `days_since_last_game` vale 1 | Recoge el salto de riesgo de jugar dos días seguidos: 1,80 % frente a 0,99 % | **Sí:** sale del calendario |
| `starter_share` | % de titularidades en los 10 últimos partidos jugados, sin incluir el actual | Rol: de 0,66 % (nunca titular) a 1,40 % (siempre titular) | **Sí:** no incluye si es titular en el partido que predigo |
| `prior_injuries_season` | Episodios de lesión anteriores en la temporada, sin contar el del propio partido | Historial: de 0,77 % (ninguna) a 4,14 % (tres o más) | **Sí:** solo cuenta lesiones anteriores |
| `is_playoffs` | 1 si el partido es de Playoffs o Play-in | Fase final: 0,28 % frente a 1,19 % | **Sí:** el tipo de partido se sabe de antemano |
| `edad` | Edad el día del partido | De 0,97 % (menos de 25) a 1,35 % (30 o más) | **Sí:** se sabe de antemano |
| `numMinutes`, `points`, tiros del partido, `is_starter` | Datos del propio partido | — | **No -> descartadas** (fuga de información) |
| TS% (eficiencia) | Puntos y tiros del propio partido | Solo para probar H2 | **No -> descartada** (fuga de información) |

**What this means for the stakeholder.** El modelo mira ocho datos de cada jugador que se conocen la víspera del partido: 
- cuántos minutos ha jugado en las últimas dos semanas, 
- cuánto jugó en el último partido, 
- cuántos días ha descansado, 
- si juega dos días seguidos, 
- si es titular, 
- cuántas lesiones lleva en la temporada, 
- si es un partido de playoffs y su edad. 

Los que más diferencia marcan son el historial de lesiones (con tres o más lesiones en la temporada, el riesgo es cinco veces mayor),la carga de las últimas dos semanas y jugar dos días seguidos (los dos casi triplican el riesgo). En cambio, jugar dos días seguidos no hace que el jugador rinda peor: el motivo para darle descanso es el riesgo de lesión, no el rendimiento.

## 5. Split and baseline

### Separación por tiempo

El modelo se usará para predecir partidos futuros, así que lo entreno con el pasado y lo evalúo con el futuro. Si mezclara temporadas al azar, el modelo aprendería con partidos posteriores a los que predice, y el resultado sería demasiado optimista.

**Decisiones:**
- **Temporadas completas, no fechas sueltas.** En la NBA, el calendario, las plantillas y las normas cambian de una temporada a otra. Cortar a mitad de temporada mezclaría partidos de la misma temporada en entrenamiento y test.
- **Train: de 2015-16 a 2020-21.** Seis temporadas para aprender. La 2021-22 no interviene porque no tiene target (no tiene comentarios). Suposición del año covid provado anteriormente. 
- **Validación: 2022-23 y 2023-24.** Para comparar modelos y elegir el umbral de alerta.
- **Test: 2024-25 y 2025-26**, las más recientes, porque son lo más parecido al futuro en el que se usaría el modelo. Uso dos temporadas para tener más lesiones y un resultado más estable. 

### Baseline: el semáforo del brief

Antes de entrenar ningún modelo necesito saber qué resultado da la regla actual, porque el modelo solo merece la pena si la mejora. La regla es el semáforo del brief: **alerta si la carga de 14 días del jugador supera el percentil 80 de su propia carga**.

**Decisiones:**
- **El umbral es personal.** Cada jugador se compara consigo mismo: 150 minutos en 14 días es mucho para un suplente y normal para una estrella. La alerta salta cuando juega más que en el 80 % de sus partidos.
- **Solo con sus partidos anteriores.** El umbral de cada partido se calcula con los partidos previos del jugador y se actualiza partido a partido, igual que funcionaría en la realidad. Si usara toda su carrera, el umbral de 2016 incluiría partidos de 2025 (fuga de información).
- **Solo los partidos que jugó.** Su carga habitual es la de cuando juega. Si contara los partidos en que estuvo lesionado (con carga baja), su umbral bajaría y el semáforo saltaría antes de tiempo al volver.
- **Mínimo de 10 partidos previos.** Con dos o tres partidos, el percentil 80 no tiene sentido. Con menos de 10, el jugador no tiene umbral y el semáforo no da alerta.

In [152]:
#PRIMERO: la carga de 14 días solo en los partidos que jugó (su carga "habitual" de trabajo)
#En los partidos que no jugó la dejo vacía, para que no baje su umbral
played_rows = df["motivo_ausencia"] == "jugó"
load_when_played = df["load_14d"].where(played_rows)

#SEGUNDO: umbral personal = percentil 80 de su carga en sus partidos ANTERIORES
#shift(1) deja fuera el partido que predigo; expanding() usa toda su historia hasta ese momento
#min_periods=10: con menos de 10 partidos previos no hay umbral
df["load_p80_player"] = load_when_played.groupby(df["personId"]).transform(
    lambda s: s.shift(1).expanding(min_periods=10).quantile(0.8)
)

#TERCERO: el semáforo: alerta (1) si la carga de 14 días supera su umbral personal
#Sin umbral (menos de 10 partidos previos), la comparación da False y no hay alerta
df["semaforo_alert"] = (df["load_14d"] > df["load_p80_player"]).astype(int)

#CUARTO: comprobación de fuga con un jugador desde el principio de los datos: Al Horford en 2015-16
check_player = df[(df["lastName"] == "Horford") & (df["season_start"] == 2015)]
check_player[["game_day", "motivo_ausencia", "load_14d", "load_p80_player", "semaforo_alert"]].head(13)

,game_day,motivo_ausencia,load_14d,load_p80_player,semaforo_alert
26518,2015-10-27,jugó,0.0,NaN,0
26519,2015-10-29,jugó,30.0,NaN,0
26520,2015-10-30,jugó,60.0,NaN,0
26521,2015-11-01,jugó,94.0,NaN,0
26522,2015-11-03,jugó,128.0,NaN,0
26523,2015-11-04,jugó,161.0,NaN,0
26524,2015-11-06,jugó,190.0,NaN,0
26525,2015-11-07,jugó,220.0,NaN,0
26526,2015-11-09,jugó,248.0,NaN,0
26527,2015-11-11,jugó,251.0,NaN,0


**Comprobación con Al Horford (inicio de 2015-16):** en sus 10 primeros partidos no tiene umbral ni alerta. En el 11.º (13/11) el umbral es el percentil 80 de sus 10 cargas anteriores, que se puede comprobar a mano: ordenadas, cae entre 220 y 248, así que 220 + 0,2 × (248 − 220) = 225,6. Su carga ese día (250) lo supera y salta la alerta. En el partido siguiente ya entra el del 13/11 en su historia y el umbral sube a 248, y como su carga (248) no lo supera, no hay alerta. El umbral nunca usa el partido del día.

**Una limitación que se ve en el ejemplo:** los primeros umbrales se calculan con el inicio de temporada, cuando la carga va subiendo desde cero, así que son bajos y el semáforo da más alertas al principio. Es como funcionaría la regla en la realidad, y el umbral se estabiliza a medida que el jugador acumula partidos.

In [153]:
#¿QUÉ TAL FUNCIONA EL SEMÁFORO?
from sklearn.metrics import precision_score, recall_score

#PRIMERO: filas del modelo y cuántas no tienen umbral
at_risk_rows = df[df["at_risk"]]
y_true = at_risk_rows["lesion_nueva"].astype(int)
y_alert = at_risk_rows["semaforo_alert"]
print("Filas en riesgo sin umbral (menos de 10 partidos previos):", at_risk_rows["load_p80_player"].isna().sum())

#SEGUNDO: rendimiento del semáforo
alert_rate = y_alert.mean()
semaforo_recall = recall_score(y_true, y_alert)
semaforo_precision = precision_score(y_true, y_alert)
print(f"\nPartidos con alerta: {alert_rate:.1%}")
print(f"Recall (lesiones nuevas detectadas): {semaforo_recall:.1%}")
print(f"Precisión (alertas que acaban en lesión): {semaforo_precision:.2%}")
print(f"Tasa de lesión sin alerta: {y_true[y_alert == 0].mean():.2%} · con alerta: {y_true[y_alert == 1].mean():.2%}")

#TERCERO: ¿es mejor que alertar al azar?
#Si diera las mismas alertas al azar, detectaría el mismo % de lesiones que de partidos alerta, y su precisión sería la tasa general
print(f"\nAl azar, con el mismo número de alertas: recall {alert_rate:.1%} · precisión {y_true.mean():.2%}")

Filas en riesgo sin umbral (menos de 10 partidos previos): 10961

Partidos con alerta: 24.7%
Recall (lesiones nuevas detectadas): 26.8%
Precisión (alertas que acaban en lesión): 1.22%
Tasa de lesión sin alerta: 1.10% · con alerta: 1.22%

Al azar, con el mismo número de alertas: recall 24.7% · precisión 1.13%


**¿Qué tal funciona el semáforo?** Mal. Pone alerta en el 24,7 % de los partidos en riesgo, uno de cada cuatro, y detecta el 26,8 % de las lesiones nuevas (recall). Si pusiera el mismo número de alertas al azar, detectaría el 24,7 %. Es decir, el semáforo solo detecta 2,1 puntos más de lesiones que el azar.

Lo mismo se ve en la precisión: de cada 100 alertas, solo 1,22 acaban en lesión, frente al 1,13 de alertar al azar. Los partidos con alerta tienen un riesgo (1,22 %) casi igual que los partidos sin alerta (1,10 %).

Además, hay 10.961 filas en riesgo (el 4 %) en las que el semáforo no puede juzgar, porque el jugador tiene menos de 10 partidos previos.

**Por qué funciona tan mal si la carga sí importa.** En el apartado 4 vi que el riesgo casi se triplica entre los jugadores con menos y con más carga. Pero el semáforo usa un solo dato y un solo corte, y compara a cada jugador consigo mismo, así que no tiene en cuenta el descanso, el historial de lesiones, la edad ni el rol del jugador. Es lo que el modelo tiene que mejorar, combinando las ocho features.

**Es el listón del modelo:** detectar más del 26,8 % de las lesiones nuevas con un número de alertas parecido, o detectar las mismas con menos alertas.

In [154]:
#GUARDAR nba_model.parquet
#PRIMERO: las columnas que guardo
#No todas son features: las de identificación sirven para analizar resultados (Celtics, temporadas, jugadores)
id_columns = ["personId", "firstName", "lastName", "gameId", "gameDate", "season_start", "playerteamName", "gameType"]
baseline_columns = ["load_p80_player", "semaforo_alert"]   #el semáforo del brief (baseline)
columns_to_save = id_columns + [TARGET] + NUMERIC + ["grupo_edad"] + baseline_columns

#SEGUNDO: solo las filas en riesgo, que son las que tienen target
model_data = df.loc[df["at_risk"], columns_to_save].copy()
#En estas filas el target nunca está vacío, así que lo paso a número entero (0/1)
model_data[TARGET] = model_data[TARGET].astype(int)

#TERCERO: guardo el archivo (ruta relativa al notebook, que está en project/)
MODEL_DATA_PATH = Path("data") / "nba_model.parquet"
model_data.to_parquet(MODEL_DATA_PATH, index=False)
size_mb = MODEL_DATA_PATH.stat().st_size / 1e6
print(f"Guardado {MODEL_DATA_PATH}: {len(model_data):,} filas, {model_data.shape[1]} columnas, {size_mb:.1f} MB")

#CUARTO: lo vuelvo a leer para comprobar que se ha guardado bien
check_saved = pd.read_parquet(MODEL_DATA_PATH)
print("Lesiones nuevas en el archivo:", check_saved[TARGET].sum(), f"({check_saved[TARGET].mean():.2%})")
print("Vacíos por columna:")
print(check_saved.isna().sum()[check_saved.isna().sum() > 0])
check_saved.dtypes

Guardado data\nba_model.parquet: 257,888 filas, 20 columnas, 3.2 MB
Lesiones nuevas en el archivo: 2905 (1.13%)
Vacíos por columna:
load_p80_player    10961
dtype: int64


personId                          int64
firstName                           str
lastName                            str
gameId                            int64
gameDate                 datetime64[us]
season_start                      int32
playerteamName                      str
gameType                            str
lesion_nueva                      int64
load_14d                        float64
minutes_last_game               float64
days_since_last_game            float64
back_to_back                      int64
starter_share                   float64
prior_injuries_season             int64
is_playoffs                       int64
edad                            float64
grupo_edad                     category
load_p80_player                 float64
semaforo_alert                    int64
dtype: object

**Dataset del modelo guardado:** `data/nba_model.parquet`, con 257.888 filas, 20 columnas y 3,2 MB. Lo leen los apartados siguientes y el notebook de la lesson 09.

**Decisiones:**
- **Solo guardo las filas en riesgo**, que son las únicas con target. Las demás solo servían para calcular la carga y las features, y eso ya está hecho.
- **Guardo columnas que no son features**, para analizar los resultados: quién es el jugador y de qué equipo (para evaluar a los Celtics aparte), la fecha y la temporada (para separar por tiempo), el grupo de edad (para comparar errores por grupo) y el semáforo (el baseline). El modelo solo usará las 8 columnas de la lista `NUMERIC`.
- **El target se guarda como número entero (0/1):** en las filas en riesgo nunca está vacío.

**Comprobación:** al volver a leer el archivo, tiene las 2.905 lesiones nuevas (1,13 %) y los tipos de datos correctos (fechas como fechas, el grupo de edad como categoría). El único vacío es el umbral del semáforo en los 10.961 partidos de jugadores con menos de 10 partidos previos, que es lo esperado.

In [155]:
#SEPARACIÓN DE TRAIN, VALIDACIÓN Y TEST
#PRIMERO: separo por temporadas completas (la 2021-22 ya no está: no tiene target)
#Train = el pasado; validación = las dos temporadas siguientes; test = las dos más recientes (no se toca hasta el apartado 8)
train = model_data[model_data["season_start"] <= 2020]
valid = model_data[model_data["season_start"].isin([2022, 2023])]
test = model_data[model_data["season_start"] >= 2024]

#SEGUNDO: features (X) y target (y) de cada parte
X_train, y_train = train[NUMERIC + CATEGORICAL], train[TARGET]
X_valid, y_valid = valid[NUMERIC + CATEGORICAL], valid[TARGET]
X_test, y_test = test[NUMERIC + CATEGORICAL], test[TARGET]

#TERCERO: resumen de las tres partes
split_summary = pd.DataFrame({
    name: {
        "Temporadas": seasons,
        "Desde": part["gameDate"].min().date(),
        "Hasta": part["gameDate"].max().date(),
        "Filas": len(part),
        "% de filas": round(len(part) / len(model_data) * 100, 1),
        "Lesiones nuevas": int(part[TARGET].sum()),
        "Tasa de lesión (%)": round(part[TARGET].mean() * 100, 2),
    }
    for name, seasons, part in [
        ("Train", "2015-16 a 2020-21", train),
        ("Validación", "2022-23 y 2023-24", valid),
        ("Test", "2024-25 y 2025-26", test),
    ]
}).T
print(split_summary)

#CUARTO: compruebo que las tres partes no se solapan en el tiempo y que no se pierde ninguna fila
print("\nSin solapamiento:", train["gameDate"].max() < valid["gameDate"].min() and valid["gameDate"].max() < test["gameDate"].min())
print("Filas en total:", len(train) + len(valid) + len(test), "| filas del modelo:", len(model_data))

#QUINTO: filas y lesiones de los Celtics en cada parte (los evaluaré aparte)
for name, part in [("Train", train), ("Validación", valid), ("Test", test)]:
    celtics = part[part["playerteamName"] == "Celtics"]
    print(f"Celtics en {name}: {len(celtics):,} filas · {int(celtics[TARGET].sum())} lesiones nuevas")

                   Temporadas       Desde       Hasta   Filas % de filas  \
Train       2015-16 a 2020-21  2015-10-28  2021-07-20  152699       59.2   
Validación  2022-23 y 2023-24  2022-10-20  2024-06-17   52008       20.2   
Test        2024-25 y 2025-26  2024-10-24  2026-06-13   53181       20.6   

           Lesiones nuevas Tasa de lesión (%)  
Train                 1533                1.0  
Validación             501               0.96  
Test                   871               1.64  

Sin solapamiento: True
Filas en total: 257888 | filas del modelo: 257888
Celtics en Train: 5,568 filas · 30 lesiones nuevas
Celtics en Validación: 1,900 filas · 11 lesiones nuevas
Celtics en Test: 1,816 filas · 55 lesiones nuevas


**Resultado:** train tiene 152.699 filas (59,2 %), validación 52.008 (20,2 %) y test 53.181 (20,6 %). Las tres partes no se solapan en el tiempo y suman las 257.888 filas del modelo.

**El test es más difícil que la validación.** La tasa de lesión nueva es del 1,00 % en train y del 0,96 % en validación, pero del 1,64 % en test. Es el aumento de las últimas temporadas que vi en el apartado 3. El modelo aprenderá con un mundo en el que hay menos lesiones de las que encontrará en el test, que es una situación realista: el futuro no siempre se parece al pasado. Lo tendré en cuenta al comparar los resultados de validación y test.

**Los Celtics:** tienen 5.568 filas y 30 lesiones nuevas en train, 1.900 filas y solo 11 lesiones en validación, y 1.816 filas y 55 lesiones en test. Su tasa pasa del 0,54-0,58 % al 3,03 % en test, más de cinco veces más. Con solo 11 lesiones, cualquier resultado de los Celtics en validación será muy inestable, así que los evaluaré sobre todo en el test (apartado 10), donde hay más casos.

In [156]:
#LOS BASELINES EN VALIDACIÓN
from sklearn.dummy import DummyClassifier
from sklearn.metrics import precision_score, recall_score

#PRIMERO: el baseline ingenuo: siempre predice la clase más frecuente ("no se lesiona")
naive_model = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

#SEGUNDO: las predicciones de los dos baselines en validación
baseline_predictions = {
    "Ingenuo (siempre 'no')": naive_model.predict(X_valid),
    "Semáforo (percentil 80)": valid["semaforo_alert"],
}

#TERCERO: tabla de resultados en validación
baseline_rows = []
for name, pred in baseline_predictions.items():
    baseline_rows.append({
        "Baseline": name,
        "% partidos con alerta": round(pd.Series(pred).mean() * 100, 1),
        "Recall (%)": round(recall_score(y_valid, pred) * 100, 1),
        "Precisión (%)": round(precision_score(y_valid, pred, zero_division=0) * 100, 2),
        "Accuracy (%)": round((pd.Series(pred, index=y_valid.index) == y_valid).mean() * 100, 1),
    })
baseline_table = pd.DataFrame(baseline_rows)
print(f"Tasa de lesión en validación: {y_valid.mean():.2%}")
baseline_table

Tasa de lesión en validación: 0.96%


,Baseline,% partidos con alerta,Recall (%),Precisión (%),Accuracy (%)
0,Ingenuo (siempre 'no'),0.0,0.0,0.00,99.0
1,Semáforo (percentil 80),23.8,30.3,1.23,75.8


**Los baselines en validación:**
- **El ingenuo** (siempre "no se lesiona") tiene un 99,0 % de accuracy y no detecta ninguna lesión (recall del 0 %). Es la mejor prueba de por qué no uso la accuracy: con un 0,96 % de lesiones, decir siempre que no acierta casi siempre y no sirve para nada.
- **El semáforo** pone alerta en el 23,8 % de los partidos y detecta el 30,3 % de las lesiones nuevas. Con el mismo número de alertas puestas al azar detectaría el 23,8 %, así que en validación mejora el azar en 6,5 puntos. De cada 100 alertas, solo 1,23 acaban en lesión.

**El listón para los modelos:** detectar más del 30,3 % de las lesiones nuevas con un número de alertas parecido (alrededor del 24 %), o detectar las mismas con menos alertas.

**What this means for the stakeholder.** Para saber si el modelo funcionaría en el futuro, lo entreno con las temporadas de 2015 a 2021 y lo pruebo en temporadas que no ha visto. La regla actual, el semáforo de carga, avisa en uno de cada cuatro partidos y solo detecta tres de cada diez lesiones nuevas: apenas mejor que avisar al azar. Es el listón que el modelo tiene que superar. Además, en las dos últimas temporadas hay muchas más lesiones que antes, también en los Celtics, así que la prueba final será más exigente.

## 6. Candidate models

Comparo dos modelos con los baselines del apartado 5: uno simple y uno de conjunto.

| Modelo | Papel | Por qué |
|---|---|---|
| **Árbol de decisión** (3 niveles) | Simple | Es un semáforo con más condiciones, aprendidas de los datos. Sus reglas se pueden leer y explicar al cuerpo técnico tal cual |
| **Gradient boosting** | De conjunto | Combina cientos de árboles pequeños, cada uno corrigiendo los errores del anterior. Ordena mejor el riesgo, pero es más difícil de explicar |

Elijo el árbol como modelo simple, y no la regresión logística, porque da el mismo tipo de respuesta que el semáforo (reglas de sí o no) y encaja con la regla de decisión de minutos: cada grupo del árbol puede ser normal, limitar o descansar. Elijo `HistGradientBoostingClassifier` y no un random forest porque es mucho más rápido con 150.000 filas de entrenamiento, y el notebook tiene que ejecutarse en menos de 10 minutos.

**Decisiones:**
- **Cada modelo va en un pipeline**, para que el preprocesado aprenda solo con train y se aplique igual a validación y test.
- **El preprocesado solo rellena vacíos con la mediana.** Los dos modelos son de árboles: deciden con cortes ("¿carga > 87?"), así que no hace falta escalar las features. Ahora no hay vacíos, pero protege si llega un jugador sin edad.
- **El árbol tiene 3 niveles y al menos 500 partidos por grupo.** Tres niveles dan como mucho 8 grupos de jugadores: suficientes para combinar varias features y pocos para poder leer las reglas. Con 500 partidos por grupo, ninguna regla se basa en unos pocos casos.
- **El gradient boosting va con sus valores por defecto.** Primero veo cuánto da tal cual; el ajuste de hiperparámetros lo hago en el notebook de la lesson 09.
- **`class_weight="balanced"` en los dos**, para que no ignoren las lesiones, que son el 1 % de los casos.
- **`random_state` fijo**, para que los resultados sean reproducibles.

**Cómo los comparo:** con `class_weight="balanced"`, cada modelo da alerta cuando su probabilidad pasa del 50 %, lo que produce un número de alertas parecido al del semáforo. Así puedo comparar cuántas lesiones detecta cada uno con un esfuerzo parecido para el cuerpo técnico. La PR-AUC mide además lo bien que ordena el riesgo, sin depender del umbral; el umbral definitivo lo elijo en el apartado 7.

In [157]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

#El procesado común: solo rellenar vacíos con la mediana. 
#Los os modelos son de árboles: deciden con cosrtes (carga > 87?),así que no hace falta escalar las features
#Ahora no hay vacíos, pero protege si llega un jugador sin edad; al ir en el pipeline, la mediana se calcula solo con train
preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUMERIC),
])

In [161]:
from sklearn.ensemble import HistGradientBoostingClassifier 
from sklearn.tree import DecisionTreeClassifier

model_candidates = {
    #MODELO SIMPLE:un árbol de 3 niveles (como mucho 8 grupos), con al menos 500 partidos por grupo
    #Es un semáforo con más condiciones, aprendidas de los datos: sus reglas se pueden leer
    "Árbol de decisión": Pipeline([
        ("prep", preprocess),
        ("model", DecisionTreeClassifier(
            max_depth=3, min_samples_leaf=500, class_weight="balanced", random_state= RANDOM_STATE
        )),  
    ]),
    #MODELO MÁS COMPLEJO: cintos de árboles pequeños, cada uno corrige los errores del anterior
    "Gradient boosting": Pipeline([
        ("prep", preprocess),
        ("model", HistGradientBoostingClassifier(class_weight="balanced", random_state= RANDOM_STATE)),
    ]),

}

In [162]:
import time
from sklearn.metrics import average_precision_score, precision_score, recall_score

#PRIMERO: entreno cada modelo con train y lo evalúo en validación
comparison_rows = []
valid_proba = {}   #guardo las probabilidades para el gráfico y para el apartado 7
for name, pipe in model_candidates.items():
    start = time.time()
    pipe.fit(X_train, y_train)
    train_seconds = time.time() - start

    proba = pipe.predict_proba(X_valid)[:, 1]   #probabilidad de lesión nueva
    alert = pipe.predict(X_valid)                #su sí/no por defecto (probabilidad > 50 %)
    valid_proba[name] = proba
    comparison_rows.append({
        "Modelo": name,
        "% partidos con alerta": round(alert.mean() * 100, 1),
        "Recall (%)": round(recall_score(y_valid, alert) * 100, 1),
        "Precisión (%)": round(precision_score(y_valid, alert) * 100, 2),
        "PR-AUC": round(average_precision_score(y_valid, proba), 4),
        "Segundos de entrenamiento": round(train_seconds, 1),
    })

#SEGUNDO: añado los baselines del apartado 5 (no dan probabilidades, así que no tienen PR-AUC)
for name, pred in baseline_predictions.items():
    comparison_rows.insert(0 if "Ingenuo" in name else 1, {
        "Modelo": name,
        "% partidos con alerta": round(pd.Series(pred).mean() * 100, 1),
        "Recall (%)": round(recall_score(y_valid, pred) * 100, 1),
        "Precisión (%)": round(precision_score(y_valid, pred, zero_division=0) * 100, 2),
        "PR-AUC": np.nan,
        "Segundos de entrenamiento": np.nan,
    })

comparison = pd.DataFrame(comparison_rows)

print(f"Tasa de lesión en validación: {y_valid.mean():.2%} (es la PR-AUC de un modelo que no sabe nada)")
comparison

Tasa de lesión en validación: 0.96% (es la PR-AUC de un modelo que no sabe nada)


,Modelo,% partidos con alerta,Recall (%),Precisión (%),PR-AUC,Segundos de entrenamiento
0,Ingenuo (siempre 'no'),0.0,0.0,0.00,NaN,NaN
1,Semáforo (percentil 80),23.8,30.3,1.23,NaN,NaN
2,Árbol de decisión,20.8,41.7,1.93,0.0166,0.2
3,Gradient boosting,25.4,49.3,1.87,0.0285,1.4


**Resultado en validación:**

- **El árbol supera al semáforo con menos alertas:** avisa en el 20,8 % de los partidos (frente al 23,8 %) y detecta el 41,7 % de las lesiones nuevas (frente al 30,3 %). De cada 100 alertas, 1,93 acaban en lesión, frente a 1,23 del semáforo.
- **El gradient boosting es el que más detecta:** con un 25,4 % de alertas, un número parecido al del semáforo, detecta el 49,3 % de las lesiones nuevas, casi la mitad.
- **El gradient boosting también ordena mejor el riesgo:** su PR-AUC es de 0,0285, casi el triple de la de un modelo que no sabe nada (0,0096). La del árbol es de 0,0166, porque solo distingue unos pocos grupos de jugadores.
- **Los dos son rápidos:** el árbol se entrena en 0,2 segundos y el gradient boosting en 1,4, así que no comprometen el límite de 10 minutos.

**En resumen:** los dos modelos mejoran claramente la regla actual. Pero la precisión sigue siendo baja en todos (alrededor de 2 alertas de cada 100 acaban en lesión): predecir una lesión concreta es muy difícil, porque muchas no dependen de la carga (golpes, caídas). El modelo sirve para **ordenar a los jugadores por riesgo** y decidir a quién cuidar, no para adivinar quién se lesionará.

In [164]:
#LAS REGLAS DEL ÁRBOL
from sklearn.tree import export_text

#PRIMERO: las reglas que ha aprendido el árbol, en texto
tree_pipe = model_candidates["Árbol de decisión"]
tree_model = tree_pipe.named_steps["model"]
print(export_text(tree_model, feature_names=NUMERIC))

#SEGUNDO: cada grupo final (hoja) del árbol en validación: cuántos partidos, su tasa de lesión y si da alerta
leaf_id = tree_model.apply(tree_pipe.named_steps["prep"].transform(X_valid))
tree_leaves = pd.DataFrame({
    "leaf": leaf_id, "alert": tree_pipe.predict(X_valid), "injury": y_valid.to_numpy(),
}).groupby("leaf").agg(rows=("injury", "size"), injury_rate=("injury", "mean"), alert=("alert", "first"))
tree_leaves["injury_rate"] = (tree_leaves["injury_rate"] * 100).round(2)
tree_leaves.sort_values("injury_rate", ascending=False)

|--- prior_injuries_season <= 0.50
|   |--- load_14d <= 87.50
|   |   |--- starter_share <= 0.27
|   |   |   |--- class: 0
|   |   |--- starter_share >  0.27
|   |   |   |--- class: 0
|   |--- load_14d >  87.50
|   |   |--- is_playoffs <= 0.50
|   |   |   |--- class: 0
|   |   |--- is_playoffs >  0.50
|   |   |   |--- class: 0
|--- prior_injuries_season >  0.50
|   |--- is_playoffs <= 0.50
|   |   |--- prior_injuries_season <= 1.50
|   |   |   |--- class: 1
|   |   |--- prior_injuries_season >  1.50
|   |   |   |--- class: 1
|   |--- is_playoffs >  0.50
|   |   |--- edad <= 25.60
|   |   |   |--- class: 0
|   |   |--- edad >  25.60
|   |   |   |--- class: 0



,rows,injury_rate,alert
leaf,,,
11,2897,2.93,1
10,7932,1.56,1
6,24517,0.91,0
13,390,0.77,0
14,1160,0.69,0
4,2107,0.66,0
7,1008,0.40,0
3,11997,0.33,0


**Las reglas del árbol.** El árbol divide los partidos en 8 grupos. Leído de arriba abajo:

| Grupo | Condiciones | Partidos (validación) | Tasa de lesión nueva | Alerta |
|---|---|---|---|---|
| 11 | 2 o más lesiones previas en la temporada · temporada regular | 2.897 | **2,93 %** | **Sí** |
| 10 | 1 lesión previa · temporada regular | 7.932 | **1,56 %** | **Sí** |
| 6 | Sin lesiones previas · carga de 14 días > 87,5 minutos · temporada regular | 24.517 | 0,91 % | No |
| 13 | Con lesiones previas · fase final · 25,6 años o menos | 390 | 0,77 % | No |
| 14 | Con lesiones previas · fase final · más de 25,6 años | 1.160 | 0,69 % | No |
| 4 | Sin lesiones previas · carga ≤ 87,5 · titular en más del 27 % de sus partidos | 2.107 | 0,66 % | No |
| 7 | Sin lesiones previas · carga > 87,5 · fase final | 1.008 | 0,40 % | No |
| 3 | Sin lesiones previas · carga ≤ 87,5 · casi siempre suplente | 11.997 | 0,33 % | No |

**La regla de alerta del árbol es muy simple: "alerta si el jugador ya se ha lesionado al menos una vez esta temporada y no es un partido de fase final".** Con dos o más lesiones previas, el riesgo es casi el triple que la media de validación (2,93 % frente a 0,96 %).

**Qué dice sobre la carga (H1).** La primera pregunta del árbol no es la carga, sino el historial de lesiones. La carga aparece en el segundo nivel, y entre los jugadores sin lesiones previas separa riesgos distintos: 0,33 % con poca carga y poco rol de titular, y 0,91 % con más de 87,5 minutos en 14 días, casi el triple. Así que la carga sí se asocia al riesgo (coherente con H1), pero por sí sola no lleva a ningún grupo por encima del umbral de alerta. **El historial de lesiones avisa mejor que la carga**, que es justo lo que le faltaba al semáforo.

**Una nota sobre la fase final:** los grupos de playoffs con lesiones previas se dividen por edad, pero son grupos pequeños (390 y 1.160 partidos), así que no les doy mucho peso.

In [167]:
#LA CURVA DE CAPTURA DEL GRADIENT BOOSTING
#¿Cuántas lesiones detecta el gradient boosting según el % de partidos con alerta?
#Ordeno los partidos de más a menos riesgo y voy acumulando las lesiones detectadas
gb_order = np.argsort(-valid_proba["Gradient boosting"])
captured = np.cumsum(y_valid.to_numpy()[gb_order]) / y_valid.sum() * 100
alert_share = np.arange(1, len(captured) + 1) / len(captured) * 100
capture_curve = pd.DataFrame({"alert_share": alert_share, "captured": captured}).iloc[::100]   #un punto de cada 100, para que el gráfico sea ligero

fig = px.line(
    capture_curve, x="alert_share", y="captured",
    title="Lesiones nuevas detectadas según el % de partidos con alerta (validación)",
    labels={"alert_share": "% de partidos con alerta", "captured": "% de lesiones nuevas detectadas"},
)
fig.data[0].name = "Gradient boosting"
fig.data[0].showlegend = True
#Los dos que dan un sí/no fijo aparecen como puntos
for row in comparison[comparison["Modelo"].isin(["Semáforo (percentil 80)", "Árbol de decisión"])].itertuples():
    fig.add_scatter(x=[row[2]], y=[row[3]], mode="markers", marker_size=12, name=row.Modelo)
#La línea del azar: con un X % de alertas al azar se detecta un X % de las lesiones
fig.add_scatter(x=[0, 100], y=[0, 100], mode="lines", line_dash="dash", name="Al azar")
fig.show()

print(f"Gradient boosting con el mismo % de alertas que el semáforo: {np.interp(23.8, alert_share, captured):.1f} % de lesiones detectadas")

Gradient boosting con el mismo % de alertas que el semáforo: 47.1 % de lesiones detectadas


**La curva de captura.** Ordeno los partidos de más a menos riesgo según el gradient boosting y miro qué porcentaje de las lesiones nuevas se detecta al dar alerta a un porcentaje de partidos. La línea discontinua es el azar.

- **El gradient boosting queda siempre por encima del azar.** Con un 5 % de alertas detecta alrededor del 18 % de las lesiones; con un 10 %, alrededor del 29 %; con un 25 %, el 49 %; y con un 50 %, el 71 %.
- **El semáforo queda claramente por debajo de la curva:** con un 23,8 % de alertas detecta el 30,3 % de las lesiones nuevas, mientras que el gradient boosting, con el mismo número de alertas, detecta el 47,1 %: 16,8 puntos más con el mismo esfuerzo para el cuerpo técnico.

**Qué significa:** con un número de alertas parecido al del semáforo, las ocho reglas del árbol detectan casi lo mismo que el gradient boosting. La ventaja del gradient boosting está en la **flexibilidad**: el árbol solo puede dar alerta al 20,8 % de los partidos o a ninguno, mientras que con el gradient boosting puedo elegir cuántas alertas dar. Si el cuerpo técnico solo puede gestionar pocas alertas (por ejemplo, un 5 o un 10 % de los partidos), el gradient boosting le dice cuáles son las más arriesgadas. Esa elección del número de alertas es la del apartado 7.

**What this means for the stakeholder.** Los dos modelos avisan mejor que el semáforo actual. Una regla aprendida de los datos ("alerta si el jugador ya se ha lesionado esta temporada, salvo en playoffs") avisa en menos partidos que el semáforo y detecta cuatro de cada diez lesiones nuevas, frente a tres de cada diez. Un modelo más complejo detecta casi la mitad con un número de avisos parecido, y además permite elegir cuántos avisos dar: si el cuerpo técnico solo puede cuidar a unos pocos jugadores, le dice cuáles son los de más riesgo. La señal más fuerte no es la carga de minutos, sino haber tenido ya una lesión esa temporada.

## 7. Metric and cost of errors

> 📝 Why is your main metric the right one for this decision? What does one error cost the
> stakeholder?
> - **Classification:** the confusion matrix read in business terms (what a false positive and
>   a false negative cost), and the threshold you chose.
> - **Regression:** MAE against RMSE, and where the large errors fall (which segments, which
>   periods).

In [ ]:
# TODO the error analysis for the best candidate

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 8. Validation and tuning

> 📝 Cross-validate the chosen model (`TimeSeriesSplit` when your data has dates), tune it with
> a grid or random search, and only then evaluate it **once** on the test set. Report the
> score per fold, not only the mean: a model that is good on average but erratic is a risk.

In [ ]:
# TODO cross-validation and the search
# cv = TimeSeriesSplit(n_splits=5)
# search = GridSearchCV(candidates["TODO"], param_grid={"model__max_depth": [4, 8, None]},
#                       cv=cv, scoring="neg_mean_absolute_error")
# search.fit(pd.concat([X_train, X_valid]), pd.concat([y_train, y_valid]))
# search.best_params_, -search.cv_results_["mean_test_score"]

In [ ]:
# TODO the one final evaluation on the test set
# final_model = search.best_estimator_
# X_test, y_test = test[NUMERIC + CATEGORICAL], test[TARGET]
# mean_absolute_error(y_test, final_model.predict(X_test))

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 9. Interpretation

> 📝 Which features drive the predictions? Use permutation importance (or SHAP if you want to
> go further). Then explain two or three individual predictions: why did the model say this for
> this row? Do the drivers match your hypotheses?

In [ ]:
# TODO feature importance
# result = permutation_importance(final_model, X_test, y_test, n_repeats=10,
#                                 random_state=RANDOM_STATE)
# importance = pd.Series(result.importances_mean, index=X_test.columns).sort_values()
# px.bar(importance, orientation="h", title="TODO")

In [ ]:
# TODO two or three individual predictions, explained

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 10. Fairness, ethics and risks

> 📝 Compare the errors across at least one meaningful group (district, segment, period): is the
> model worse for some of them? Then answer in prose:
> - Does the data contain personal or sensitive information? How is it protected (lesson 03:
>   GDPR, anonymisation, access)?
> - What happens if the model is wrong, and who bears the cost?
> - One mitigation for each risk you name.

In [ ]:
# TODO the error per group
# errors = test.assign(abs_error=(y_test - final_model.predict(X_test)).abs())
# errors.groupby("TODO_group")["abs_error"].mean()

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 11. Recommendation

> 📝 Speak to the stakeholder:
> - What should they do with the model, starting when?
> - The expected improvement over today's rule, translated into a KPI of your brief or into
>   money.
> - When it should **not** be trusted (periods, segments, data changes).
> - What block 4 adds: how it will be deployed and monitored.

*TODO*

## 12. Saving the model

> 📝 Save the final pipeline — from raw columns to a prediction — and prove it works by reloading
> it and predicting one row. This is the file lesson 10 deploys. Commit it if it is under 50 MB.

In [ ]:
# TODO save, reload, predict one row
# joblib.dump(final_model, MODEL_PATH)
# reloaded = joblib.load(MODEL_PATH)
# reloaded.predict(X_test.head(1))

## 13. References


Basketball Reference. (s. f.). Glossary. https://www.basketball-reference.com/about/glossary.html

ABC7 News. (2023, 13 de septiembre). How the NBA’s new rules on resting stars will work. https://abc7news.com/post/how-the-nbas-new-rules-on-resting-stars-will-work/13777093/

ClickOnDetroit. (2024, 4 de enero). NBA fines Nets $100,000 for violating player participation policy by resting players. https://www.clickondetroit.com/sports/2024/01/04/nba-fines-nets-100000-for-violating-player-participation-policy-by-resting-players/

ABC News. (2024, 26 de noviembre). *NBA says Hawks violated player participation policy by sitting Trae Young for Cup game*. Associated Press. https://abcnews.go.com/Sports/wireStory/nba-hawks-violated-player-participation-policy-sitting-trae-116250982

Basketball Reference. (s. f.). *Glossary*. Recuperado el 13 de noviembre de 2026, de https://www.basketball-reference.com/about/glossary.html

ClickOnDetroit. (2024, 4 de enero). *NBA fines Nets $100,000 for violating player participation policy by resting players*. Associated Press. https://www.clickondetroit.com/sports/2024/01/04/nba-fines-nets-100000-for-violating-player-participation-policy-by-resting-players/

Fox Sports. (2025, 12 de marzo). *NBA fines Jazz $100,000 for holding Lauri Markkanen out of recent games*. Associated Press. https://foxsports.com/articles/nba/nba-fines-jazz-100000-for-holding-lauri-markkanen-out-of-recent-games

McKinney, W. (2010). Data structures for statistical computing in Python. En S. van der Walt y J. Millman (Eds.), *Proceedings of the 9th Python in Science Conference* (pp. 56–61). https://doi.org/10.25080/Majora-92bf1922-00a

Moore, E. A. (2026). *NBA Dataset: Box Scores and Stats (1947 - Today)* (Versión 515) [Conjunto de datos]. Kaggle. Recuperado el 14 de octubre de 2026, de https://www.kaggle.com/datasets/eoinamoore/historical-nba-data-and-player-box-scores

NBA. (2024). *Brooklyn Nets fined $100,000*. NBA Official. https://official.nba.com/brooklyn-nets-fined-100000/

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M. y Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.

WTOP. (2025, noviembre). *NBA fines Cavaliers $100,000 for resting All-Stars Mitchell and Mobley*. Associated Press. https://wtop.com/sports/2025/11/nba-fines-cavaliers-100000-for-resting-all-stars-mitchell-and-mobley
